# Scaling of Social Connections & Demographic Analysis
---
#### Ayush Sarkar: 5/15/2025 - 7/11/2025 | CLIMA w/ Dynamical Systems Lab @ NYU
*A study of how Facebook-derived social connectivity scales with population across U.S. counties and Core-Based Statistical Areas (CBSAs), paired with Census-block demographics of coastal NYC communities.*

---

# Objectives:

* **CiviL Infrastructure research for climate change Mitigation and Adaptation (CLIMA)** is a research effort focused on infrastructure research to help develop equitable and feasible solutions to the increasingly urgent threats *posed* by climate change through mitigating damages and adapting to hazards and changes across coastal communities.

* This leg of CLIMA is an interdisciplinary research project that aims to model the effects of flood risk on coastal communities through a detailed investigation of social networks, alongside other factors, in an attempt to create models that are more capable of capturing human mobility phenomena, especially amongst homeowners.

* By using a modified compartmental model, we can split the population into unique sets and utilize the mean-field hypothesis to treat individuals as identical, allowing us to focus on the population's dynamics instead of individual-level cognition. To better appreciate the unique perspectives of individuals and communities that face these threats, CLIMA also seeks to include qualitative information about the network extracted through interviews with homeowners of coastal communities here in NYC.

* Additionally, demographic breakdowns of the locations of study were done to further understand the context and everyday lives of the homeowners within these communities. On the quantitative side, we utilize more traditional methods, including historical Census data and related public data, to build a more comprehensive demographic profile.

* This **notebook specifically focuses on (1) the scaling of social connections with population** across counties and CBSAs, and **(2) demographic analyses** of the coastal NYC communities under study (with Hamilton Beach as the running example, and Red Hook as a comparison).

---

# List of Tasks:

1. Completed CITI training
2. Completed interview transcriptions
3. Computed scaling exponents for social connections vs. population at different administrative divisions in the US, including:
    * County-level
    * CBSA-level (combined Metropolitan + Micropolitan)
    * MSA-level (Metropolitan Statistical Area)
    * muSA-level (Micropolitan Statistical Area)
4. Attempted to perform community identification using Clique Percolation:
    * Unsuccessful due to hardware and time constraints on a ~10M-row symmetric edge list
5. Generated demographic analyses for the relevant coastal NYC communities (Hamilton Beach, Howard Beach, Red Hook)

---

# Table of Contents:

1. **Introduction:**
    * Datasets and their provenance
    * Preliminary notation and definitions
    * Network types (county- and CBSA-level)
    * SCI and connection equations
    * Urban scaling background
    * Improving $\beta$ via rescaling (per *Schläpfer et al.*)
2. **Data Cleaning & Exploration:**
    * Loading and cleaning the SCI, ESRI, crosswalk, and Census files
    * Choropleth maps for geographic visualization
    * Histograms with KDE and parametric PDF fits
3. **Scaling Results:**
    * Log-log linear regression plots at the County, CBSA, MSA, and muSA levels
    * Tabulated $\beta$ and $\gamma$ estimates with confidence intervals
4. **Demographic Analyses (Census Block resolution, NYC):**
    * Age/Sex population pyramids w/ NYC averages
    * Race/Ethnicity distribution w/ NYC averages
    * Persons per household (1-5+) distribution w/ NYC averages
    * Housing occupancy status distribution w/ NYC averages
    * Housing tenure (owner vs. renter) distribution w/ NYC averages
    * Pie charts of educational attainment and household income for Hamilton Beach and Red Hook
5. **Discussion, future work, and references**

---

# Quantitative Datasets:

## Datasets
1. **U.S. Bureau of Labor Statistics County-to-CBSA Crosswalk** — used to map counties to Core-Based Statistical Areas (CBSAs), and from there to MSAs vs. muSAs.
2. **ESRI 2022 Monthly Active Facebook Users and Population Estimates by County** — provides the user counts $|S_i|$ and population estimates $N_i$ that enter both the connection formulas and the coverage estimate $s_i$.
3. **Meta's Social Connectedness Index (SCI)** — the directional, county-to-county friendship-tie measure that anchors the entire connection calculation.
4. **Meta Public Coverage Estimates** — used as an external sanity check on our derived $s_i$ values.
5. **IPUMS NHGIS: Census Demographics Files at Census Block resolution** — for the demographic side of the project:
   - 2020 Decennial Census Demographic and Housing Characteristics (DHC):
     * `H3`: Housing Occupancy Counts
     * `H4`: Housing Tenure Counts
     * `H9`: Household Size Counts
     * `P3`: Race counts
     * `P5`: Hispanic or Latino origin by race
     * `P12`: Sex by age across the total population
   - `nhgis0004_ds267_20235`: Educational attainment and household income
6. **2021 County- and CBSA-level TIGER/Line shapefiles** for choropleths.

## Meta SCI

The [County-to-County data sheet](https://data.humdata.org/dataset/e9988552-74e4-4ff4-943f-c782ac8bca87/resource/c59fd5ac-0458-4e83-b6be-5334f0ea9a69/download/us-counties-us-counties-fb-social-connectedness-index-october-2021.zip) includes the following columns:

1. `user_loc` — GEOID of the source location ($i$)
2. `fr_loc` — GEOID of the target location ($j$)
3. `scaled_sci` — Scaled Social Connectedness Index ($\text{SCI}_{i,j}$)
4. Original dataframe dimensions: $[10{,}413{,}529 \times 3]$
   * After cleaning, we have 3,141 unique FIPS codes, 917 CBSA codes, 381 MSA codes, and 536 muSA codes.
     - Expected directed row count: $2 \cdot \binom{3{,}141}{2} + 3{,}141 = 9{,}865{,}881$ cleaned SCI row entries.
5. **Notes:**
   - The dataset includes both $(i,j)$ and $(j,i)$ entries (i.e., it is symmetric / fully directed).
   - SCI is reported multiplied by $10^9$, and Gaussian noise in the range $\pm[0,1]$ is added to the connection count between any GEOID pair, ensuring the minimum number of connections between two locations is non-zero. Counties with fewer than 50,000 active users are not included in the SCI file.
     - [Source: SCI Methodology](https://dataforgood.facebook.com/dfg/docs/methodology-social-connectedness-index)
   - The SCI is generally stable over time — it has been shown to predict trade flows in the 1980s about as well as it does today, and the same has been shown for mutual-fund investments in the 2000s vs. as of 2021. This stability is partly why we are comfortable pairing 2021 SCI with 2022 ESRI user / population estimates.
     - [Source: SCI Docs](https://data.humdata.org/dataset/e9988552-74e4-4ff4-943f-c782ac8bca87/resource/a0c37eb4-b45c-436d-b2b2-c0c9b1974318/download/documentation-fb-social-connectedness-index-october-2021.pdf)

## ESRI 2022 Facebook Monthly Active Users (MAU) and Population Estimates

The [ESRI dataset](https://nyuds.maps.arcgis.com/home/item.html?id=14a2fb32e22b4fe5ab9d884c9e994075) is a multipurpose collection of market potential indices for various commercial services and goods, including Facebook MAU estimates. It also includes multiple GEOID resolutions (e.g., counties, blocks, ZCTAs). The columns of interest are:

1. `MP19049a_B` — 2022 Social Media: used Facebook in the last 30 days (our $|S_i|$)
2. `TOTPOP_CY` — 2022 total population (ESRI, non-Census) (our $N_i$)
3. `ID` — 2022 5-digit FIPS county code

We choose ESRI specifically because Meta does **not** publish official county-level MAU counts; the ESRI estimates are the best publicly available stand-in, which is what makes the coverage-rescaling step in *Schläpfer et al.* necessary in our setting.

## Meta Public Coverage Estimates:

<p align='center'>
    <img src="image/public cov_ests/Screenshot 2026-01-22 at 00.57.27.png" alt="Meta Daily Active Users" width="1250"/>
</p>

<p align='center'>
    <img src="image/public cov_ests/Screenshot 2026-01-22 at 00.50.55.png" alt="Meta Monthly Active Users" width="1250"/>
</p>

* According to Meta's 2022 quarterly performance disclosures for shareholders, we can find an official estimate for the coverage rate across North America (United States + Canada). Since our ESRI data is from 2022, we focus on the entries associated with Q1-Q4 2022, where there are **199 million daily active users (DAU)** and **266 million monthly active users (MAU)** as of Q4 2022. Using 2022 population estimates — United States $\approx$ 333.3 M, Canada $\approx$ 38.94 M, summing to 372.24 M — we obtain:
   * MAU coverage estimate for North America: $266/372.24 \approx 0.715$
   * DAU coverage estimate for North America: $199/372.24 \approx 0.535$
* These figures bracket what we should *expect* for our county-level coverage estimates $s_i$. As we'll see in the choropleth section, our ESRI-derived $s_i$ values average around 0.55 — close to the Meta DAU figure but below the MAU figure by ~17%, which we revisit in the **Potential Problems** section.

---

# Preliminary Notation and Definitions:

A brief glossary so that the formulas below are easy to read. Throughout, "GEOID" refers generically to a geographic unit — usually a county (5-digit FIPS) or a CBSA (5-digit CBSA code). The subscript $i$ indexes an individual GEOID; the subscript on $k$ tags the *type* of degree being summed.

$$
   \begin{align*}
        &{\large \textbf{Social Network Graph Notation}}\\
        &G = \text{Whole social network graph (all counties, all edges)} \\
        &C = \text{GEOID-level covering (e.g., the set of counties forming a single CBSA)} \\
        &\tilde{C} = \text{Complement covering (all counties outside } C \text{)} \\
        &n_{\text{GEOID}} = \text{Number of GEOID-level nodes} \in G \\\\
        &{\large \textbf{Degree and Connection Notation}}\\
        &k_{i,\,o} = \text{Outgoing degree of GEOID } i \text{ (cross-GEOID connections)} \\
        &k_{i,\,ic} = \text{Inter-GEOID degree of GEOID } i \text{ (within-GEOID connections)} \\
        &k_{i,\,t} = \text{Total degree of GEOID } i = k_{i,\,o} + k_{i,\,ic} \\
        &K_{r,\,i} = \text{Rescaled cumulative degree of } i \text{ (i.e., } k_{i,\,t}/s_i \text{)} \\
        &\langle K_{r}\rangle = \text{Average rescaled cumulative degree over all GEOIDs} \\\\
        &{\large \textbf{User and Population Notation}}\\
        &N_i = \text{Population estimate of GEOID } i \text{ (ESRI 2022)} \\
        &\langle N\rangle = \text{Average GEOID population estimate} \\
        &|S_i| = \text{2022 Facebook MAU estimate (30-day) for GEOID } i \\
        &s_i = |S_i| / N_i = \text{Coverage estimate for GEOID } i \\\\
        &{\large \textbf{Scaling Notation}}\\
        &\beta = \text{Scaling exponent (slope in log-log space)} \\
        &\gamma = \text{Intercept (in log-log space)}
    \end{align*}
$$

> **A note on terminology:** in this notebook, "**Inter-**" is used to denote *within-GEOID* connections (i.e., $(i,i)$ pairs), while "**Outer-**" / "**Outgoing**" denote *cross-GEOID* connections (i.e., $(i,j)$ with $i \neq j$). This is non-standard — the prefix "inter-" conventionally means *between* — but it is internally consistent across the codebase, dataframe column names, and figures. If you are reading the variable `inter_county_connections`, that is the within-county degree.

---

# Network Types:
---
## County-level network types
At the **county level**, we partition connections into three disjoint types — **Inter**, **Outgoing**, and **Total** (= Inter + Outgoing). For the county-level graph $G_{\text{County}}$ and a fixed home county $i$:

- **Inter-county (within-county) connections**

  $$
  E_{ic}(i) = \{(u, v) \mid u, v \in i,\ u \neq v\} \tag{Inter-County Conn.}
  $$
  * Connections between two users $u, v$ who both reside in county $i$. These are loops on the county node when the graph is contracted to county granularity.

- **Outgoing (cross-county) connections**

  $$
  E_o(i) = \{(u, v) \mid u \in i,\ v \in j,\ j \neq i\} \tag{Outer-County Conn.}
  $$
  * Connections between a user $u$ in county $i$ and a user $v$ in some other county $j$.

- **Total county connections**

  $$
  E_t(i) = E_{ic}(i) \;\cup\; E_o(i) \tag{Total County Conn.}
  $$
  * Every connection touching county $i$.

The cumulative degree $k_{i,\,t}$ is simply $|E_t(i)|$, and likewise for $k_{i,\,ic}$ and $k_{i,\,o}$.

---
## CBSA-level network types
At the **CBSA level**, the picture is one nesting deeper, because each CBSA is itself a covering $C$ over multiple counties. We define four CBSA-level types: **ICIC**, **ICCC**, **OCOC**, and **Total** (= ICIC + ICCC + OCOC).

- **Inter-covering inter-county (ICIC) — within-CBSA, within-county**

  $$
  \textbf{ICIC}(C) = \bigcup_{i \in C} E_{ic}(i) \tag{ICIC CBSA Conn.}
  $$
   * Connections that live inside a single county *and* whose county sits inside the CBSA covering. Summed across all counties in $C$.

- **Inter-covering cross-county (ICCC) — within-CBSA, between-county**

  $$
  \textbf{ICCC}(C) = \bigcup_{\substack{i,\,j \in C \\ i \neq j}} E_o(i)\big|_{j} \tag{ICCC CBSA Conn.}
  $$
   * Connections between two different counties $i \neq j$, both of which sit inside the same CBSA covering.

- **Outer-covering cross-county (OCOC) — outside-CBSA connections**

  $$
  \textbf{OCOC}(C) = \bigcup_{\substack{i \in C \\ j \in \tilde{C}}} E_o(i)\big|_{j} \tag{Outer-CBSA Conn.}
  $$
   * Connections between a county $i$ inside the CBSA covering and a partner county $j$ outside it. This captures the "external reach" of a CBSA.

- **Total CBSA connections**

  $$
  \textbf{Total}(C) = \textbf{ICIC}(C) \,\cup\, \textbf{ICCC}(C) \,\cup\, \textbf{OCOC}(C) \tag{Total CBSA Conn.}
  $$
   * The full degree of the CBSA in $G$. As we'll see in the next section, the fact that these three sets are *disjoint* is what lets us recover OCOC by subtraction instead of enumeration.

---

# Important Equations:
---
As a quick preface, the SCI equations come from Meta's [Social Connectedness Index Project documentation](https://data.humdata.org/dataset/e9988552-74e4-4ff4-943f-c782ac8bca87/resource/a0c37eb4-b45c-436d-b2b2-c0c9b1974318/download/documentation-fb-social-connectedness-index-october-2021.pdf). From these, we invert to solve for Facebook connection counts at the county level and then aggregate up to CBSAs.

  - **Combinatorial refresher**
        $$
                \binom{n}{k}
                := \frac{n!}{k!(n-k)!}, \quad n,k \in \mathbb{N}
                \tag{Combinations}
        $$
    - A Facebook connection involves exactly **two users**, so we set $k = 2$. The combination formula then answers: *given $n$ objects, how many unique unordered pairs can we form?*

    - There are two places this matters in our setup:

      **(1) Within-county user pairs.** With $|S_i|$ users in county $i$, the number of *ordered* (user, other user) pairs is $|S_i| \cdot (|S_i| - 1)$, exactly twice the unordered count $\binom{|S_i|}{2}$. Meta's homogeneous SCI formula uses the *ordered* count in its denominator, which is why the within-county connection equation below has $|S_i|(|S_i|-1)$ rather than $\binom{|S_i|}{2}$ — each Facebook friendship contributes one edge in each direction.

      **(2) Cross-county GEOID pairs in the SCI dataset.** Since $\text{SCI}_{i,j}$ is indexed by GEOIDs $i$ and $j$, the same accounting determines the *number of rows* in the SCI dataset. With $n_{\text{GEOID}}$ counties, symmetric off-diagonal entries (both $(i,j)$ and $(j,i)$ are recorded), and self-pairs $(i,i)$:

        $$
                \text{SCI Row Entries}
                = 2 \cdot \binom{n_{\text{GEOID}}}{2} + n_{\text{GEOID}}
                = n_{\text{GEOID}}^2
                \tag{Total SCI Row Entries}
        $$

      For $n_{\text{GEOID}} = 3{,}141$ cleaned U.S. counties, this gives $9{,}865{,}881$ rows, matching what we get after the cleaning step.

## SCI Equations:
---
  - **Homogeneous SCI (within-county)** — represents the average connection density among ordered user pairs in a single county:
        $$
         \text{SCI}_{i,i}
        = \frac{\text{FB Conn.}_{i,i}}
        {|S_i| \cdot (|S_i| - 1)}
        \tag{Homogeneous SCI}
        $$

 - **Heterogeneous SCI (cross-county)** — represents the average connection density between two different counties:
        $$
        \text{SCI}_{i,j}
        = \frac{\text{FB Conn.}_{i,j}}
        {|S_i| \cdot |S_j|}
        \tag{Heterogeneous SCI}
        $$

We *invert* these two relations to solve for the connection counts, which is what we actually want to study.

## Connections Equations:
---
* ### County-level connections:
   - **Inter-county Facebook connections**
                $$
                        \text{FB Conn.}_{i,i}
                        = \text{SCI}_{i,i}
                        \cdot \big(|S_i|
                        \cdot (|S_i| - 1)\big)
                        \tag{Inter-County Conn.}
                $$

   - **Outer-county (cross-county) Facebook connections**
                $$
                        \text{FB Conn.}_{i,j}
                        = \text{SCI}_{i,j}
                        \cdot \big(|S_i| \cdot |S_j|\big)
                        \tag{Outgoing County Conn.}
                $$

   - **Total county-level Facebook connections**
                $$
                        \begin{align}
                        \text{FB Conn.}_i
                        &= \text{FB Conn.}_{i,i}
                        + \sum_{\substack{j \in G_{\text{County}} \\ j \neq i}}
                        \text{FB Conn.}_{i,j} \\
                        &= \text{SCI}_{i,i}
                        \cdot \big(|S_i|
                        \cdot (|S_i| - 1)\big)
                        + \sum_{\substack{j \in \text{Counties} \\ j \neq i}}
                        \text{SCI}_{i,j} \cdot |S_i| \cdot |S_j|
                        \end{align}
                        \tag{Total County Conn.}
                $$
---
* ### CBSA-level connections:
SCI is only defined at county-to-county granularity, so the CBSA-level connection counts are *built up* by summing county-level connections across the counties belonging to each CBSA.

   - **Inter-covering inter-county CBSA-level connections (ICIC)**
                $$
                        \textbf{ICIC}_{C}
                        = \sum_{i \in C}
                        \text{SCI}_{i,i}
                        \cdot |S_i| \cdot (|S_i| - 1)
                        \tag{ICIC Conn.}
                $$

    - **Inter-covering cross-county CBSA-level connections (ICCC)**
                $$
                        \textbf{ICCC}_{C}
                        = \sum_{\substack{i, j \in C \\ i \neq j}}
                        \text{SCI}_{i,j}
                        \cdot |S_i| \cdot |S_j|
                        \tag{ICCC Conn.}
                $$

   - **Outer-covering CBSA-level connections (OCOC)**
                $$
                        \textbf{OCOC}_{C}
                        = \sum_{\substack{i \in C,\ j \in \tilde{C}}}
                        \text{SCI}_{i,j}
                        \cdot |S_i| \cdot |S_j|
                        \tag{Outer CBSA Conn.}
                $$

   - **Total CBSA-level Facebook connections**
                $$
                        \textbf{Total}_{C}
                        = \textbf{ICIC}_{C} + \textbf{ICCC}_{C} + \textbf{OCOC}_{C}
                        \tag{Total CBSA Conn.}
                $$
---

## Symmetry, SCI, and Connections:
- Since SCI is not defined at the CBSA level, we compute CBSA-level total connections by summing the total connections of all counties within the CBSA covering. This is valid because the total number of connections in the graph is **invariant under topological coverings** — that is, the number of connections does not change whether we consider county-level or CBSA-level partitions.

- This invariance lets us exploit the fact that, while the *distribution* of connections across county GEOIDs changes when we coarse-grain to CBSAs, the *total number of connections in $G$ is conserved*. As a result, we can algebraically recover **OCOC** in terms of **ICIC**, **ICCC**, and **Total**, rather than computing $\textbf{OCOC}_{C}$ directly.

  This approach is necessary due to the symmetry of FIPS code pairs in the SCI dataset:

  - Attempting to *unsymmetrize* the SCI dataset using the `user_loc` or `fr_loc` columns — by removing duplicate FIPS-FIPS pairs in either direction — introduces a bias toward whichever county's FIPS code happens to appear first in the dataframe (typically the lower-numbered one). This process also effectively converts the graph into an undirected one, since only $(i,j)$ would remain and not $(j,i)$. Depending on downstream merges or indexing operations, this can lead to significant issues. For this reason, **we keep the SCI dataset symmetric throughout the pipeline**.

  - Instead of iterating over the full dataframe and matching FIPS-FIPS pairs explicitly, we group counties by their CBSA codes and sum the total county-level connections (inter + outgoing) for all counties within each CBSA covering. From the disjoint partition $\textbf{Total}_{C} = \textbf{ICIC}_{C} + \textbf{ICCC}_{C} + \textbf{OCOC}_{C}$, we obtain:

  $$
  \textbf{OCOC}_{C}
  = \textbf{Total}_{C}
  - \big[\textbf{ICIC}_{C} + \textbf{ICCC}_{C}\big]
  \tag{OCOC}
  $$
---

## Power-Law Scaling Relationships

* We begin by assuming a power-law relationship between the cumulative degree $k_{i,\,t}$ and the GEOID population $N_i$. This is the same functional form that has been observed empirically for face-to-face interactions in cities, GDP, patent counts, and a wide range of other urban metrics:

$$
k_{i,\,t} = N_i^{\beta} \cdot \epsilon_i
\tag{Basic Power Law}
$$

* Applying a log-log transformation (and treating $\epsilon_i$ as multiplicative noise so that $\log \epsilon_i$ is additive on the log scale), we proceed as:
$$
\begin{align}
    \log(k_{i,\,t}) &= \log(N_{i}^{\beta}) + \log(\epsilon_i) \\\\
              &= \beta \cdot \log(N_i) + \log(\epsilon_i)
\end{align}
\tag{Log-Linear Form}
$$

* To recover the scaling exponent $\beta$ and the intercept $\gamma$, we plot each county/GEOID as a single point in the plane defined by two GEOID features (e.g., the number of connections vs. population, or — in classical allometry — metabolic rate vs. body mass).
* By plotting GEOID feature $y$ vs. GEOID feature $x$ for all available GEOIDs and fitting a regression, we are identifying a *national-level macroscopic trend* between two aggregate features — each county contributes one data point.
* We work in log-log space because logarithms convert proportional differences into additive ones, which is the natural representation for the multiplicative processes that pervade biology and the social sciences, and because they make the data scale-invariant: a doubling of population looks the same whether the baseline is 10,000 or 10,000,000.


* We then estimate the relationship using **linear regression**, producing a scatter plot with a line of best fit. The slope of this line, $\beta$, characterizes the power-law scaling relationship between cumulative degree and population:

  - **Superlinear:** $\beta > 1$ — bigger places generate *disproportionately* more connections per person (typical of within-city interaction, innovation, GDP)
  - **Linear:** $\beta = 1$ — connections scale exactly with population (per-capita rate is constant)
  - **Sublinear:** $\beta < 1$ — bigger places have *proportionally fewer* connections per person (typical of infrastructure economies of scale)

* This framework lays the foundation for understanding how social connectivity scales with population across counties.
---

## Improving $\beta$ via Rescaling

Following [*Schläpfer et al.*](https://doi.org/10.1098/rsif.2013.0789), we improve estimation of the scaling exponent $\beta$ by rescaling the cumulative degree $k_{i,\,t}$:

1. **Divide cumulative degree by the coverage estimate** of location $i$, denoted $s_i$. The motivation: our $k_{i,\,t}$ only counts connections among the Facebook-active subset of residents, so a county with low coverage will look artificially under-connected. Dividing by $s_i$ corrects for this sampling bias and recovers a quantity proportional to the true population-level degree.
2. **Normalize both population and rescaled cumulative degree by their averages**, which tightens the linear fit and raises the $R^2$ — geometrically, this re-centers the cloud of points on $(1, 1)$ so that the regression isn't sensitive to absolute units.
---
### County-level rescaling

We first apply this methodology at the county level. Since the SCI data are already at county-to-county resolution, no initial aggregation step is required.

1. **Compute county coverage estimates**

   $$
   s_{\text{County}_i}
   = \frac{|S_i|}{N_i},
   \quad \forall i \in G
   $$

2. **Rescale cumulative degree by coverage**

   $$
   K_{r,\,\text{County}_i}
   = \frac{k_{i,\,t}^{(\text{County})}}{s_{\text{County}_i}}
   $$

3. **Compute averages of the rescaled degree and population over all counties**

   $$
   \langle K_{r,\,\text{County}} \rangle
   = \frac{1}{n_{\text{County}}} \sum_{i=1}^{n_{\text{County}}}
     K_{r,\,\text{County}_i}
   $$

   $$
   \langle N_{\text{County}} \rangle
   = \frac{1}{n_{\text{County}}} \sum_{i=1}^{n_{\text{County}}}
     N_{\text{County}_i}
   $$

4. **Normalize variables prior to fitting**

   - Normalize population:

     $$
     N_{\text{County}_i}
     \;\leftarrow\; \frac{N_{\text{County}_i}}{\langle N_{\text{County}} \rangle}
     \tag{County Population Normalization}
     $$

   - Normalize rescaled cumulative degree:

     $$
     K_{r,\,\text{County}_i}
     \;\leftarrow\; \frac{K_{r,\,\text{County}_i}}
            {\langle K_{r,\,\text{County}} \rangle}
     \tag{Rescaled County Degree}
     $$

5. **Estimate $\beta$**

   Each county is plotted as a point in the plane of log-transformed, normalized, rescaled cumulative degree vs. log-transformed, normalized population. An OLS fit (`statsmodels`) yields a line of best fit whose slope $\beta$ characterizes the scaling relationship:

   - **Superlinear:** $\beta > 1$
   - **Linear:** $\beta = 1$
   - **Sublinear:** $\beta < 1$

---
### CBSA-level rescaling

The same procedure carries over to CBSAs once we've aggregated county-level user and population estimates up to the CBSA covering.

1. **Aggregate county-level user and population estimates**

   $$
   |S|_{\text{CBSA}}
   = \sum_{i \in C_{\text{CBSA}}} |S_i|,
   \qquad
   N_{\text{CBSA}}
   = \sum_{i \in C_{\text{CBSA}}} N_i
   $$

2. **Compute CBSA coverage estimates**

   $$
   s_{\text{CBSA}}
   = \frac{|S|_{\text{CBSA}}}{N_{\text{CBSA}}}
   $$

3. **Rescale CBSA cumulative degree**

   $$
   K_{r,\,\text{CBSA}}
   = \frac{k_t^{(\text{CBSA})}}{s_{\text{CBSA}}}
   $$

4. **Compute averages of the rescaled degree and population over all CBSAs**

   $$
   \langle K_{r,\,\text{CBSA}} \rangle
   = \frac{1}{n_{\text{CBSA}}}\sum_{i=1}^{n_{\text{CBSA}}}
     K_{r,\,\text{CBSA}_i}
   $$

   $$
   \langle N_{\text{CBSA}} \rangle
   = \frac{1}{n_{\text{CBSA}}}\sum_{i=1}^{n_{\text{CBSA}}}
     N_{\text{CBSA}_i}
   $$

5. **Normalize variables prior to fitting** — analogous to steps 4a/4b above, but with CBSA averages.

6. **Estimate $\beta$**

   Each CBSA is plotted as a point in the plane of log-transformed, normalized, rescaled cumulative degree vs. log-transformed, normalized population. The OLS fit yields a best-fit line whose slope $\beta$ characterizes the scaling at the CBSA level. When we compute MSA-only and muSA-only fits, we re-take the averages using *only* MSAs (or *only* muSAs) so that the normalization is internal to each subpopulation and not contaminated by the size mismatch between metropolitan and micropolitan areas.

---

# Code:

The remainder of this notebook is the executable implementation of the framework above. Cells are organized so that:

1. **Requirements** — libraries and raw datasets are loaded once.
2. **Data Cleaning** — the SCI, ESRI, and crosswalk files are merged into the four export dataframes (`df_outer_county`, `df_cbsa`, `df_msa`, `df_musa`) that all downstream sections consume.
3. **Data Exploration** — histograms with KDE/parametric fits and choropleth maps inspect the distributions and geographic structure of connections.
4. **Scaling** — log-log OLS regressions produce the $\beta$ estimates summarized in the results table at the bottom of the notebook.
5. **Demographics** — Census-block-level analyses of the Hamilton Beach (and comparison) communities.

The scripts under `scripts/` mirror these cells one-to-one and can be run standalone if you prefer not to re-execute the notebook.



## Requirements:

`pandas`, `numpy`, `matplotlib`, `geopandas`, `scipy`, `statsmodels`, and `scikit-learn`. The choropleth section additionally uses `esda` for Moran's I (commented out by default), and reads TIGER/Line shapefiles via `geopandas.read_file`. Memory note: the raw SCI TSV is ~180 MB and decompresses to a ~10M-row dataframe — be mindful when running this on a constrained machine.

### Libraries:
All the third-party packages we'll need across data cleaning, statistical fitting, and plotting. We import them up front so each downstream cell can be run in isolation after the kernel restarts.

In [ ]:
"""Library imports used throughout the notebook.

All third-party packages needed across data cleaning, statistical
fitting, and plotting are imported up front so each downstream cell can
be run in isolation after a kernel restart.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import geopandas as gpd
import math
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import lognorm, skewnorm, genpareto, norm, gaussian_kde
from matplotlib.ticker import FixedLocator, FixedFormatter
from matplotlib.colors import Normalize

### Data Sets:
Load every raw source in one place so the rest of the notebook is decoupled from file paths. `dtype=str` is used wherever a column will eventually be a FIPS code or GEOID, because pandas would otherwise drop the leading zeros that matter for joining (e.g., `01001` for Autauga County, AL).

In [ ]:
"""Load every raw input in one place so the rest of the notebook is
decoupled from file paths. ``dtype=str`` is used wherever a column will
eventually be a FIPS or GEOID code, because pandas would otherwise drop
the leading zeros that matter for joining (e.g. ``01001`` for Autauga
County, AL).
"""
df_sci = pd.read_table('F:\\dsl_CLIMA\\Social Connectdness Data\\County-County Data\\county_county.tsv', dtype=str)
df_users = pd.read_csv('F:\\dsl_CLIMA\\arcGIS User Data\\county_users.csv', dtype=str)
df_crosswalk = pd.read_excel('F:\\dsl_CLIMA\\MSAs and MicroSAs\\Census Data\\list1.xls', header=2, dtype=str)

# Decennial 2020 DHC files for the demographics section.
df_tenure = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H4-Data.csv', dtype={'GEO_ID':str, 'NAME':str})
df_household_size = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H9-Data.csv', dtype={'GEO_ID':str, 'NAME':str})
df_sex_by_age = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P12-Data.csv', dtype={'GEO_ID':str, 'NAME':str})
df_race = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P3-Data.csv', dtype={'GEO_ID':str, 'NAME':str})
df_hispanic = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P5-Data.csv', dtype={'GEO_ID':str, 'NAME':str})
df_occupancy = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H3-Data.csv', dtype={'GEO_ID':str, 'NAME':str})

# County and CBSA basemap polygons for the choropleths.
gdf_county = gpd.read_file('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\shape files\\county\\tl_2021_us_county.shp')
gdf_cbsa = gpd.read_file('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\shape files\\cbsa\\tl_2021_us_cbsa.shp')

## Data Cleaning:

This cell takes the raw SCI, ESRI, and BLS crosswalk files and produces the four export dataframes that the rest of the notebook (and the standalone scripts) consume:

- `df_inner_county` — one row per county, with `inter_county_connections` (within), `outer_county_connections` (cross-county sum), `total connections`, plus `user_est`, `pop_est`, `coverage est`, and CBSA metadata.
- `df_cbsa` — one row per CBSA, with the three CBSA-level connection types (ICIC, ICCC, OCOC), plus aggregated `user_est` / `pop_est` and rescaled / normalized variants used for the regressions.
- `df_msa`, `df_musa` — subsets of `df_cbsa` for which the normalization averages have been recomputed *within* the metropolitan-only and micropolitan-only subpopulations, respectively, so each fit is internally consistent.

Notable manual fixes inside this cell:
- **Alaskan census-area reorganization**: `02261` (Valdez-Cordova) was split into `02063` (Chugach) and `02066` (Copper River) after the 2020 redistricting. We re-merge them into `02261` so the SCI (which still uses the pre-split codes' replacement) joins cleanly to ESRI.
- **`15005` Kalawao County, HI** is dropped — it has a population in the low double digits and breaks log-scale plots.

In [ ]:
"""
data_cleaning.py
================
Build the four export dataframes consumed by the rest of the project.

Outputs (written to ``export/``):

- ``df_outer_county.csv``: one row per cleaned FIPS county. Columns
  include within-county, cross-county, and total connection counts; the
  ESRI user-count and population estimates; the derived coverage
  estimate; CBSA metadata (code, title, Metropolitan/Micropolitan
  classification); and the rescaled / normalized variables used in the
  log-log regressions.
- ``df_cbsa.csv``: one row per CBSA. Columns include the three
  CBSA-level connection types (ICIC, ICCC, OCOC), aggregated user and
  population estimates, coverage, and rescaled / normalized variants.
- ``df_msa.csv``: subset of ``df_cbsa`` for Metropolitan Statistical
  Areas only, with normalization averages recomputed *within* the MSA
  subpopulation so each fit is internally consistent.
- ``df_musa.csv``: subset of ``df_cbsa`` for Micropolitan Statistical
  Areas only, with normalization averages recomputed within the muSA
  subpopulation.

Inputs (read from ``source/``):

- ``source/sci/county_county.tsv``: Meta Social Connectedness Index,
  October 2021, county-to-county. Columns: ``user_loc``, ``fr_loc``,
  ``scaled_sci``. Includes both (i,j) and (j,i) entries.
- ``source/users/county_users.csv``: ESRI 2022 county estimates.
  ``MP19049a_B`` is the 2022 Facebook MAU estimate, ``TOTPOP_CY`` is the
  2022 total population, ``ID`` is the 5-digit FIPS county code.
- ``source/crosswalk/list1.xls``: U.S. BLS county-to-CBSA crosswalk used
  to map each FIPS county to its CBSA, MSA, or muSA.

Method:
Connection counts are recovered from the Meta SCI formula by inverting
the published normalization. CBSA-level totals exploit the conservation
identity ``Total = ICIC + ICCC + OCOC`` so the cross-covering OCOC term
is recovered by subtraction rather than enumerated over the full ~10M-row
symmetric edge list explicitly. Coverage rescaling and normalization by
the GEOID-level averages follow Schlapfer et al. (2014).
"""

import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# Load raw inputs.
# All identifiers are read as strings so that leading zeros on FIPS codes
# (e.g. "01001" for Autauga County, AL) are preserved through joins.
# ----------------------------------------------------------------------
df_sci = pd.read_table('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\sci\\county_county.tsv', dtype=str)
df_users = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\users\\county_users.csv', dtype=str)
df_crosswalk = pd.read_excel('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\crosswalk\\list1.xls', header=2, dtype=str)

# ----------------------------------------------------------------------
# BLS crosswalk: drop the three trailing footer rows, combine the state
# and county FIPS columns into one 5-digit FIPS, and keep only the
# columns we need downstream.
# ----------------------------------------------------------------------
df_crosswalk = df_crosswalk.iloc[:-3]
df_crosswalk['FIPS'] = df_crosswalk['FIPS State Code'] + df_crosswalk['FIPS County Code']
df_crosswalk['FIPS'] = df_crosswalk['FIPS'].astype(str).str.zfill(5)
df_crosswalk = df_crosswalk[['FIPS', 'CBSA Code', 'Metropolitan/Micropolitan Statistical Area', 'CBSA Title']].copy()

# ----------------------------------------------------------------------
# ESRI users: keep only the MAU estimate, population, and FIPS columns;
# coerce to numeric; aggregate any duplicate FIPS rows by sum.
# ----------------------------------------------------------------------
df_users = df_users[['MP19049a_B', 'TOTPOP_CY', 'ID']]
df_users = df_users.rename(columns={'MP19049a_B': 'user_count', 'TOTPOP_CY': 'pop', 'ID': 'FIPS'})

df_users['FIPS'] = df_users['FIPS'].astype(str).str.zfill(5)
df_users['user_count'] = pd.to_numeric(df_users['user_count'], errors='coerce')
df_users['pop'] = pd.to_numeric(df_users['pop'], errors='coerce')

df_users = df_users.set_index('FIPS')
df_users = df_users.groupby(df_users.index)[['user_count', 'pop']].sum()

# Alaska reorganization: 02261 (Valdez-Cordova) was split into 02063
# (Chugach) and 02066 (Copper River) after the 2020 redistricting. We
# re-merge them under 02261 so the SCI (which still uses the pre-split
# replacement code) joins cleanly to ESRI.
# 15005 (Kalawao County, HI) is dropped because its population is in the
# low double digits and produces extreme outliers on log-scale plots.
df_users.loc['02261'] = df_users.loc[['02066', '02063']].sum()
df_users = df_users.drop(index=['02066', '02063', '15005'])

# ----------------------------------------------------------------------
# Restrict the SCI to cleaned FIPS pairs and recover Facebook connection
# counts. The published SCI is multiplied by 1e9, hence the division.
#
# Meta SCI formulas (inverted to solve for connection counts):
#   SCI_{i,i} = FB_Conn_{i,i} / (|S_i| * (|S_i| - 1))
#   SCI_{i,j} = FB_Conn_{i,j} / (|S_i| * |S_j|)
# ----------------------------------------------------------------------
df_sci = df_sci[df_sci['user_loc'].isin(df_users.index) & df_sci['fr_loc'].isin(df_users.index)]
df_sci['scaled_sci'] = df_sci['scaled_sci'].astype(float)
df_sci['scaled_sci'] = df_sci['scaled_sci'] / 1000000000

df_sci['user_user_count'] = df_sci['user_loc'].map(df_users['user_count'])
df_sci['user_pop'] = df_sci['user_loc'].map(df_users['pop'])
df_sci['fr_user_count'] = df_sci['fr_loc'].map(df_users['user_count'])
df_sci['fr_pop'] = df_sci['fr_loc'].map(df_users['pop'])

# Within-county rows use the homogeneous SCI denominator; cross-county
# rows use the heterogeneous denominator.
df_sci['Connections'] = np.where(
    df_sci['user_loc'] == df_sci['fr_loc'],
    df_sci['scaled_sci'] * (df_sci['user_user_count'] * (df_sci['user_user_count'] - 1)),
    df_sci['scaled_sci'] * df_sci['user_user_count'] * df_sci['fr_user_count']
)

# Attach CBSA metadata for both endpoints of each SCI row so we can group
# by CBSA later. Suffix '_fr' is the partner county.
df_county = df_sci.merge(df_crosswalk, left_on='user_loc', right_on='FIPS', how='left', suffixes=('', '_user')).copy()
df_county = df_county.merge(df_crosswalk, left_on='fr_loc', right_on='FIPS', how='left', suffixes=('', '_fr')).copy()

df_county['user_user_count'] = pd.to_numeric(df_county['user_user_count'], errors='coerce')
df_county['user_pop'] = pd.to_numeric(df_county['user_pop'], errors='coerce')

# ----------------------------------------------------------------------
# County-level aggregation.
# - df_inner_county: one row per home county with the within-county
#   connection count plus user/pop/CBSA metadata.
# - df_outer_county: sum of all cross-county connections originating in
#   the home county.
# Inner + Outer = Total county-level degree.
# ----------------------------------------------------------------------
df_inner_county = (
    df_county[df_county['user_loc'] == df_county['fr_loc']]
    .groupby('user_loc', as_index=False)
    .agg(
        user_est=('user_user_count', 'first'),
        pop_est=('user_pop', 'first'),
        metro_micro_area=('Metropolitan/Micropolitan Statistical Area', 'first'),
        CBSA_code=('CBSA Code', 'first'),
        CBSA_title=('CBSA Title', 'first'),
        inter_county_connections=('Connections', 'first'),
    )
).copy()

df_outer_county = (
    df_county[df_county['user_loc'] != df_county['fr_loc']]
    .groupby('user_loc', as_index=False)
    .agg(outer_county_connections=('Connections', 'sum'))
).copy()

df_inner_county['outer_county_connections'] = df_outer_county['outer_county_connections']
df_inner_county['total connections'] = (
    df_outer_county['outer_county_connections'] + df_inner_county['inter_county_connections']
)

# ----------------------------------------------------------------------
# CBSA-level aggregation.
#
# ICIC: within-CBSA, within-county. Sum of (i,i) connections for i in C.
# ICCC: within-CBSA, between-county. Sum of (i,j) with i != j and both
#       in the same CBSA.
# OCOC: outside-CBSA. (i,j) with i in C and j outside C.
#
# Note: when a CBSA contains exactly one county, ICCC is undefined / 0,
# which is handled by the np.where on the ``total inter_cbsa`` column.
# ----------------------------------------------------------------------
df_inter_county_inter_cbsa = (
    df_county[
        (df_county['CBSA Code'] == df_county['CBSA Code_fr']) &
        (df_county['user_loc'] == df_county['fr_loc'])
    ]
    .groupby('CBSA Code', as_index=False)
    .agg(
        CBSA_title=('CBSA Title', 'first'),
        metro_micro_area=('Metropolitan/Micropolitan Statistical Area', 'first'),
        user_est=('user_user_count', 'sum'),
        pop_est=('user_pop', 'sum'),
        inter_cbsa_connections=('Connections', 'sum'),
    )
).copy()

df_outer_county_inter_cbsa = (
    df_county[df_county['CBSA Code'] == df_county['CBSA Code_fr']]
    .query("user_loc != fr_loc")
    .groupby('CBSA Code', as_index=False)
    .agg(outer_county_inter_cbsa_connections=('Connections', 'sum'))
).copy()

df_outer_cbsa = (
    df_county[
        (df_county['CBSA Title'] != df_county['CBSA Title_fr']) &
        (df_county['user_loc'] != df_county['fr_loc'])
    ]
    .groupby('CBSA Code', as_index=False)
    .agg(outer_cbsa_connections=('Connections', 'sum'))
).copy()

df_cbsa = (
    df_inter_county_inter_cbsa
    .merge(df_outer_county_inter_cbsa, on='CBSA Code', how='left')
    .merge(df_outer_cbsa, on='CBSA Code', how='left')
)

# Single-county CBSAs have no ICCC term, so the inter-CBSA total is
# just ICIC. Multi-county CBSAs add the ICCC piece on top.
df_cbsa['total inter_cbsa connections'] = np.where(
    df_county.groupby('CBSA Code')['user_loc'].nunique() == 1,
    df_cbsa['inter_cbsa_connections'],
    df_cbsa['inter_cbsa_connections'] + df_cbsa['outer_county_inter_cbsa_connections']
)

df_cbsa['total connections'] = df_cbsa['total inter_cbsa connections'] + df_cbsa['outer_cbsa_connections']

# ----------------------------------------------------------------------
# Coverage rescaling (Schlapfer et al. 2014).
# s = |S| / N. Rescaled degree K_r = k / s recovers the population-level
# degree the SCI undercounts because it only sees the Facebook-active
# subset of residents.
# ----------------------------------------------------------------------
df_cbsa['coverage est'] = df_cbsa['user_est'] / df_cbsa['pop_est']

df_cbsa['rescaled total inter_cbsa connections'] = df_cbsa['total inter_cbsa connections'] / df_cbsa['coverage est']
df_cbsa['rescaled outer_cbsa_connections'] = df_cbsa['outer_cbsa_connections'] / df_cbsa['coverage est']
df_cbsa['rescaled total connections'] = df_cbsa['total connections'] / df_cbsa['coverage est']

# ----------------------------------------------------------------------
# Subset to MSA-only and muSA-only views. The normalization averages
# below are taken *within* each subpopulation so each regression's
# normalization is internally consistent and not contaminated by the
# size mismatch between metropolitan and micropolitan areas.
# ----------------------------------------------------------------------
df_msa = df_cbsa[df_cbsa['metro_micro_area'] == 'Metropolitan Statistical Area'].copy()
df_musa = df_cbsa[df_cbsa['metro_micro_area'] == 'Micropolitan Statistical Area'].copy()

df_msa['normed pop_est'] = df_msa['pop_est'] / df_msa['pop_est'].mean()
df_musa['normed pop_est'] = df_musa['pop_est'] / df_musa['pop_est'].mean()
df_cbsa['normed pop_est'] = df_cbsa['pop_est'] / df_cbsa['pop_est'].mean()

# Normalize each rescaled-connection column by its own mean so the
# resulting series is centered around 1 and the OLS fit becomes
# scale-invariant.
df_msa['rescaled total inter_cbsa connections'] = (
    df_msa['rescaled total inter_cbsa connections'] / df_msa['rescaled total inter_cbsa connections'].mean()
)
df_msa['rescaled outer_cbsa_connections'] = (
    df_msa['rescaled outer_cbsa_connections'] / df_msa['rescaled outer_cbsa_connections'].mean()
)
df_msa['rescaled total connections'] = (
    df_msa['rescaled total connections'] / df_msa['rescaled total connections'].mean()
)

df_musa['rescaled total inter_cbsa connections'] = (
    df_musa['rescaled total inter_cbsa connections'] / df_musa['rescaled total inter_cbsa connections'].mean()
)
df_musa['rescaled outer_cbsa_connections'] = (
    df_musa['rescaled outer_cbsa_connections'] / df_musa['rescaled outer_cbsa_connections'].mean()
)
df_musa['rescaled total connections'] = (
    df_musa['rescaled total connections'] / df_musa['rescaled total connections'].mean()
)

df_cbsa['rescaled total inter_cbsa connections'] = (
    df_cbsa['rescaled total inter_cbsa connections'] / df_cbsa['rescaled total inter_cbsa connections'].mean()
)
df_cbsa['rescaled outer_cbsa_connections'] = (
    df_cbsa['rescaled outer_cbsa_connections'] / df_cbsa['rescaled outer_cbsa_connections'].mean()
)
df_cbsa['rescaled total connections'] = (
    df_cbsa['rescaled total connections'] / df_cbsa['rescaled total connections'].mean()
)

# County-level coverage rescaling and normalization (parallel to the
# CBSA-level operations above).
df_inner_county['coverage est'] = df_inner_county['user_est'] / df_inner_county['pop_est']
df_inner_county['normed pop_est'] = df_inner_county['pop_est'] / df_inner_county['pop_est'].mean()

df_inner_county['rescaled inter_county_connections'] = (
    df_inner_county['inter_county_connections'] / df_inner_county['coverage est']
)
df_inner_county['rescaled outer_county_connections'] = (
    df_inner_county['outer_county_connections'] / df_inner_county['coverage est']
)
df_inner_county['rescaled total connections'] = (
    df_inner_county['total connections'] / df_inner_county['coverage est']
)

df_inner_county['rescaled inter_county_connections'] = (
    df_inner_county['rescaled inter_county_connections'] / df_inner_county['rescaled inter_county_connections'].mean()
)
df_inner_county['rescaled outer_county_connections'] = (
    df_inner_county['rescaled outer_county_connections'] / df_inner_county['rescaled outer_county_connections'].mean()
)
df_inner_county['rescaled total connections'] = (
    df_inner_county['rescaled total connections'] / df_inner_county['rescaled total connections'].mean()
)

# ----------------------------------------------------------------------
# Export the four working dataframes for downstream visualization
# scripts and the notebook.
# ----------------------------------------------------------------------
df_inner_county.to_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_outer_county.csv', index=False)
df_cbsa.to_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_cbsa.csv', index=False)
df_msa.to_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_msa.csv', index=False)
df_musa.to_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_musa.csv', index=False)

## Data Exploration:

Before fitting any regressions, we want to verify that the cleaned data behaves the way the literature suggests: heavy-tailed distributions of connections, roughly log-normal user/population marginals, and visible geographic structure in the connection counts. The next two subsections — histograms and choropleths — are about building that confidence.

### Distributions and Histograms:

For each of the connection / user / population / coverage variables — at the County, CBSA, MSA, and muSA levels — we plot a histogram with:

- A non-parametric **Gaussian KDE** as the empirical reference PDF.
- Parametric fits suggested by the urban-scaling literature, evaluated against that KDE via RMSE.
- The 25th and 75th percentiles shaded, plus markers for the mean and median.

The point is two-fold: (1) confirm that the variables we're about to log-transform are well-suited to a log-log fit (i.e., they look log-normal or close to it), and (2) flag any GEOIDs that sit deep in the tails so we know what's pulling the regression around.

**Quick Notes:**
  1) We choose parametric PDFs in line with the documentation of [*Schläpfer et al.*](https://doi.org/10.1098/rsif.2013.0789), and with what is typically found when describing social and urban networks:
     - **Normal** — sanity-check baseline; if the data is genuinely log-normal in the original units, the *log-transformed* histogram should look approximately Normal.
     - **Log-Normal** — the most common candidate for connection counts (multiplicative growth processes).
     - **Skew-Normal** — accommodates the moderate asymmetry typical of these distributions without the heavy tails of a Pareto.
     - **Generalized Pareto (fit on the top 75% tail, i.e., values above the 25th percentile)** — captures tail behavior without assuming the tail starts at zero.
  2) To form a non-parametric PDF of best fit for each histogram, I use a Gaussian-kernel **KDE**. I then compare the parametric PDFs above to the KDE PDF by computing the **root mean squared error (RMSE)** at the bin centers. The lowest-RMSE parametric is reported as the closest fit. This is a deliberately simple comparison — it doesn't penalize parameter count or test goodness-of-fit formally — but it is enough to flag which of the canonical PDFs is *closest* to what the data look like.

In [ ]:
"""
histograms.py
=============
Generate histogram + KDE + parametric-PDF comparison figures for the
connection / user / population / coverage variables produced by
``data_cleaning.py``.

For each variable, the plot stacks:

- A density histogram with Freedman-Diaconis bin width (capped at
  ``bin_max``).
- 25th- and 75th-percentile shading on the histogram patches so the
  central 50% of the distribution reads at a glance.
- A Gaussian-kernel KDE as the non-parametric reference PDF.
- Parametric overlays: Normal, SkewNormal, LogNormal (when the data is
  log-transformed), and a Generalized Pareto fit to the top-75% tail
  (i.e. values above the 25th percentile).
- A stats panel reporting RMSE of each parametric PDF vs. the KDE at the
  bin centers, plus the mean / median / variance of the data.

Outputs (written to ``plots/histograms/``):

- ``connectivity/sci/sci_histograms.png``: County-level SCI distributions.
- ``connectivity/connections/{county,cbsa,msa,musa}_connections_histograms.png``:
  Connection-count distributions at each GEOID level.
- ``per_capita & per_user/{county,cbsa,msa,musa}_connections_histograms.png``:
  Connections-per-capita and connections-per-user distributions.
"""

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import lognorm, skewnorm, genpareto, norm, gaussian_kde

# ----------------------------------------------------------------------
# Load cleaned dataframes plus the raw SCI table. The raw SCI is needed
# for the first histogram block (county-level SCI distributions); the
# others only need the cleaned exports.
# ----------------------------------------------------------------------
df_inner_county = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_outer_county.csv')
df_cbsa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_cbsa.csv')
df_msa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_msa.csv')
df_musa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_musa.csv')
df_sci = pd.read_table('F:\\dsl_CLIMA\\submittable\\source\\sci\\county_county.tsv', dtype={'user_loc': str, 'fr_loc': str, 'scaled_sci': int})


def near_square_grid(k, min_cols=3):
    """Return ``(rows, cols)`` for laying out ``k`` subplots near-square.

    Forces at least ``min_cols`` columns so that single-row layouts stay
    horizontally oriented.
    """
    cols = max(min_cols, math.ceil(np.sqrt(k)))
    rows = math.ceil(k / cols)
    return rows, cols


def plot_histograms_with_pdfs_kde(
    dfs, cols, titles=None, x_labels=None, log_x=False, no_log_cols=None,
    figsize_per_plot=4, bar_padding=0.03,
    title_fontsize=12, stats_fontsize=10, legend_fontsize=9,
    box_alpha=0.15,
    bin_max=None,
    kde_max=None
):
    """Plot density histograms with KDE + parametric overlays in a grid.

    Each subplot takes one ``(df, col)`` pair from ``dfs`` and ``cols``,
    builds a density histogram of ``df[col]``, overlays a Gaussian KDE,
    and fits a panel of parametric PDFs (Normal, SkewNormal, optionally
    LogNormal, and a Generalized Pareto for the top tail). Each
    parametric is scored against the KDE by RMSE at the bin centers and
    the scores are written into a per-subplot stats box.

    Parameters
    ----------
    dfs : list of pandas.DataFrame
        One source dataframe per subplot.
    cols : list of str
        Column to plot from each dataframe.
    titles, x_labels : list of str, optional
        Per-subplot titles and x-axis labels. Default to ``cols``.
    log_x : bool, default False
        If True, log10-transform each column before binning.
    no_log_cols : list of str or int, optional
        Column names or indices for which to override ``log_x=True`` to
        False (e.g. the linear coverage column when everything else is
        log-transformed).
    figsize_per_plot : float
        Inches allocated per subplot.
    bar_padding : float
        Vertical padding for the horizontal separator lines between
        subplot rows.
    title_fontsize, stats_fontsize, legend_fontsize : int
        Font sizes for title, stats box, and legend respectively.
    box_alpha : float
        Alpha for the stats-box background.
    bin_max : int, optional
        Cap on Freedman-Diaconis bin count.
    kde_max : int, optional
        If the data exceeds ``kde_max`` points, subsample to that size
        before fitting the KDE (the KDE fitter scales poorly otherwise).

    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : ndarray of matplotlib.axes.Axes
    """
    if titles is None:
        titles = cols
    if x_labels is None:
        x_labels = cols

    k = len(cols)
    rows, cols_n = near_square_grid(k, min_cols=3)

    fig, axes = plt.subplots(
        rows, cols_n,
        figsize=(cols_n * figsize_per_plot * 1.3, rows * figsize_per_plot * 1.15),
        squeeze=False,
        constrained_layout=True
    )
    axes = axes.flatten()

    for idx, (ax, df_sub, col, title, xlabel) in enumerate(
        zip(axes, dfs, cols, titles, x_labels)
    ):

        # ---------- Data handling ----------
        # Decide whether this column gets log-transformed; ``no_log_cols``
        # lets the caller override on a per-column basis (e.g. coverage).
        raw_data = df_sub[col].dropna().values

        use_log = log_x
        if no_log_cols is not None:
            if col in no_log_cols or idx in no_log_cols:
                use_log = False

        if use_log:
            raw_data = raw_data[raw_data > 0]
            data = np.log10(raw_data)
        else:
            data = raw_data

        if len(data) < 5:
            ax.axis('off')
            continue

        mean_val = np.mean(data)
        median_val = np.median(data)
        variance_val = np.var(data)
        q25, q75 = np.percentile(data, [25, 75])

        # ---------- Freedman-Diaconis bins (capped at ``bin_max``) ----------
        h = 2 * (q75 - q25) / len(data) ** (1 / 3)
        if h <= 0 or not np.isfinite(h):
            bins = 'auto'
        else:
            bins = int(np.ceil((data.max() - data.min()) / h))

        if bin_max is not None:
            bins = min(bins, bin_max)

        counts, bin_edges, patches = ax.hist(
            data,
            bins=bins,
            density=True,
            alpha=0.6,
            color='skyblue',
            edgecolor='black'
        )

        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # ---------- Percentile shading on the histogram patches ----------
        for i, patch in enumerate(patches):
            if bin_centers[i] <= q25:
                patch.set_facecolor('lightcoral')
                patch.set_alpha(0.4)
            elif bin_centers[i] >= q75:
                patch.set_facecolor('plum')
                patch.set_alpha(0.4)

        # ---------- KDE (subsampled for tractability) ----------
        if kde_max is not None and len(data) > kde_max:
            kde_data = np.random.choice(data, kde_max, replace=False)
        else:
            kde_data = data

        kde = gaussian_kde(kde_data)
        x_kde = np.linspace(data.min(), data.max(), 500)
        y_kde = kde(x_kde)
        ax.plot(x_kde, y_kde, color='black', lw=2, label='KDE')

        # ---------- Parametric fits ----------
        fitted_pdfs = {}

        mu, sigma = norm.fit(data)
        fitted_pdfs['Normal'] = norm.pdf(bin_centers, mu, sigma)

        a, loc_sn, scale_sn = skewnorm.fit(data)
        fitted_pdfs['SkewNormal'] = skewnorm.pdf(bin_centers, a, loc_sn, scale_sn)

        # LogNormal only makes sense on log-transformed data; the
        # change-of-variables factor np.log(10) * 10**bin_centers maps
        # the PDF back to the log10 axis.
        if use_log:
            shape, loc, scale = lognorm.fit(raw_data, floc=0)
            fitted_pdfs['LogNormal'] = (
                lognorm.pdf(10**bin_centers, shape, loc, scale)
                * np.log(10)
                * 10**bin_centers
            )

        # ---------- Tail detection and Generalized Pareto fit ----------
        # Fit GenPareto on the top 75% tail (values above the 25th
        # percentile). ``tail_side`` picks which side of the plot has
        # more headroom so the legend / stats box doesn't collide with
        # the tail.
        tail_q = 25
        u = np.percentile(raw_data, tail_q)
        tail_raw = raw_data[raw_data > u] - u

        u_plot = np.log10(u) if use_log else u
        dist_left = u_plot - data.min()
        dist_right = data.max() - u_plot
        tail_side = 'right' if dist_right < dist_left else 'left'

        if len(tail_raw) > 10:
            c, loc_gp, scale_gp = genpareto.fit(tail_raw, floc=0)

            tail_mask = (10**bin_centers > u) if use_log else (bin_centers > u)
            x_tail = (
                10**bin_centers[tail_mask] - u
                if use_log else
                bin_centers[tail_mask] - u
            )

            pdf_gp = (
                genpareto.pdf(x_tail, c, loc_gp, scale_gp)
                * (np.log(10) * 10**bin_centers[tail_mask] if use_log else 1.0)
            )

            gp_full = np.zeros_like(bin_centers)
            gp_full[tail_mask] = pdf_gp
            fitted_pdfs['GenPareto (25% Tail)'] = gp_full

        # ---------- Plot all parametric PDFs ----------
        for name, pdf in fitted_pdfs.items():
            ax.plot(bin_centers, pdf, lw=2, label=name)

        # ---------- Goodness-of-fit (RMSE vs. KDE at bin centers) ----------
        kde_vals = kde(bin_centers)
        metrics = {
            name: np.sqrt(np.mean((pdf - kde_vals) ** 2))
            for name, pdf in fitted_pdfs.items()
        }

        stats_lines = ["RMSE of KDE vs PDF Estimates:"]
        for name, rmse in sorted(metrics.items(), key=lambda x: x[1]):
            stats_lines.append(f"{name}: {rmse:.4f}")

        stats_lines.append(f"\nMean: {mean_val:.4f}")
        stats_lines.append(f"Median: {median_val:.4f}")
        stats_lines.append(f"Variance: {variance_val:.4f}")

        ax.scatter(mean_val, kde(mean_val), color='red', s=80, zorder=5, label='Mean')
        ax.scatter(median_val, kde(median_val), color='darkgreen', s=80, zorder=5, label='Median')

        # ---------- Legend and stats panel placement ----------
        if tail_side == 'right':
            stats_x, stats_ha = 0.02, 'left'
            legend_loc = 'upper left'
        else:
            stats_x, stats_ha = 0.98, 'right'
            legend_loc = 'upper right'

        hist_proxy = Patch(
            facecolor='skyblue',
            edgecolor='black',
            alpha=0.6,
            label='Histogram (25-75%)'
        )

        handles, labels = ax.get_legend_handles_labels()

        handles = (
            [hist_proxy]
            + handles
            + [
                Patch(facecolor='lightcoral', edgecolor='black', label='<=25th percentile'),
                Patch(facecolor='plum', edgecolor='black', label='>=75th percentile')
            ]
        )

        legend = ax.legend(
            handles=handles,
            fontsize=legend_fontsize,
            loc=legend_loc,
            frameon=True
        )

        fig = ax.figure
        fig.canvas.draw()
        legend_bbox = legend.get_window_extent().transformed(ax.transAxes.inverted())
        stats_y = legend_bbox.y0 - 0.04

        ax.text(
            stats_x,
            stats_y,
            "\n".join(stats_lines),
            transform=ax.transAxes,
            fontsize=stats_fontsize,
            va='top',
            ha=stats_ha,
            bbox=dict(facecolor='white', alpha=box_alpha, edgecolor='black')
        )

        # ---------- Axis labels ----------
        ax.set_title(title, fontsize=title_fontsize, pad=5)
        ax.set_xlabel(f"$\log_{{10}}$({xlabel})" if use_log else xlabel, fontsize=12)
        if use_log:
            ax.set_ylabel(r'Density per $\log_{10}$(unit)', fontsize=12)
        else:
            ax.set_ylabel('Density')

    for ax in axes[k:]:
        ax.axis('off')

    # ---------- Horizontal separators between subplot rows ----------
    for row in range(1, rows):
        axes_above = axes[(row - 1) * cols_n: row * cols_n]
        y_bottom = min(ax.get_position().y0 for ax in axes_above)
        line = Line2D(
            [0, 1],
            [y_bottom - bar_padding - 0.0255],
            transform=fig.transFigure,
            color='black',
            linewidth=1.5,
            clip_on=False
        )
        fig.add_artist(line)

    return fig, axes


# ----------------------------------------------------------------------
# Connectivity and user-count histograms.
# ----------------------------------------------------------------------

# County-level SCI histograms.
# Dedup the directed SCI by (min, max) of the pair so each county pair
# contributes once; then split into within-county (endo) and cross-county
# (exo) subsets for the per-network-type panels.
df_dedup = (
    df_sci
    .assign(
        loc_min=np.minimum(df_sci['user_loc'], df_sci['fr_loc']),
        loc_max=np.maximum(df_sci['user_loc'], df_sci['fr_loc'])
    )
    .drop_duplicates(subset=['loc_min', 'loc_max'])
    .drop(columns=['loc_min', 'loc_max'])
)
df_endo = (df_dedup[df_dedup['user_loc'] == df_dedup['fr_loc']]).copy()
df_exo = (df_dedup[df_dedup['user_loc'] != df_dedup['fr_loc']]).copy()

fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_endo, df_exo, df_dedup],
    cols=['scaled_sci', 'scaled_sci', 'scaled_sci'],
    titles=[
        '$\log_{10}$(Inner-County SCI) per County',
        '$\log_{10}$(Outgoing County SCI) per County',
        '$\log_{10}$(Total County SCI) per County'
    ],
    x_labels=['Inner-County SCI', 'Outgoing County SCI', 'Total County SCI'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9,
    bin_max=65,
    kde_max=200000
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\connectivity\\sci\\sci_histograms.png", dpi=800, bbox_inches='tight')

# County-level connection histograms (six panels: three connection types
# plus user, population, and coverage marginals).
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_inner_county] * 6,
    cols=['inter_county_connections', 'outer_county_connections', 'total connections', 'user_est', 'pop_est', 'coverage est'],
    titles=[
        '$\log_{10}$(Inner-County Connections) per County',
        '$\log_{10}$(Outgoing County Connections) per County',
        '$\log_{10}$(Total County Connections) per County',
        '$\log_{10}$(User Estimate) per County',
        '$\log_{10}$(Population Estimate) per County',
        'Coverage Estimate per County'
    ],
    x_labels=['Inner-County Connections', 'Outgoing County Connections', 'Total County Connections',
              'County User Estimates', 'County Population Estimates', 'County Coverage Estimates'],
    no_log_cols=['coverage est'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\connectivity\\connections\\county_connections_histograms.png", dpi=800, bbox_inches='tight')

# CBSA-level connection histograms (combined Metropolitan + Micropolitan).
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['total inter_cbsa connections', 'outer_cbsa_connections', 'total connections',
          'user_est', 'pop_est', 'coverage est'],
    titles=[
        '$\log_{10}$(Inner-CBSA Connections) per CBSA',
        '$\log_{10}$(Outgoing CBSA Connections) per CBSA',
        '$\log_{10}$(Total CBSA Connections) per CBSA',
        '$\log_{10}$(User Estimate) per CBSA',
        '$\log_{10}$(Population Estimate) per CBSA',
        'Coverage Estimate per CBSA'
    ],
    x_labels=['Inner-CBSA Connections', 'Outgoing CBSA Connections', 'Total CBSA Connections',
              'CBSA User Estimates', 'CBSA Population Estimates', 'CBSA Coverage Estimates'],
    no_log_cols=['coverage est'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\connectivity\\connections\\cbsa_connections_histograms.png", dpi=800, bbox_inches='tight')

# MSA-only connection histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['total inter_cbsa connections', 'outer_cbsa_connections', 'total connections',
          'user_est', 'pop_est', 'coverage est'],
    titles=[
        '$\log_{10}$(Inner-MSA Connections) per MSA',
        '$\log_{10}$(Outgoing MSA Connections) per MSA',
        '$\log_{10}$(Total MSA Connections) per MSA',
        '$\log_{10}$(User Estimate) per MSA',
        '$\log_{10}$(Population Estimate) per MSA',
        'Coverage Estimate per MSA'
    ],
    x_labels=['Inner-MSA Connections', 'Outgoing MSA Connections', 'Total MSA Connections',
              'MSA User Estimates', 'MSA Population Estimates', 'MSA Coverage Estimates'],
    no_log_cols=['coverage est'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\connectivity\\connections\\msa_connections_histograms.png", dpi=800, bbox_inches='tight')

# muSA-only connection histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['total inter_cbsa connections', 'outer_cbsa_connections', 'total connections',
          'user_est', 'pop_est', 'coverage est'],
    titles=[
        '$\log_{10}$(Inner-muSA Connections) per muSA',
        '$\log_{10}$(Outgoing muSA Connections) per muSA',
        '$\log_{10}$(Total muSA Connections) per muSA',
        '$\log_{10}$(User Estimate) per muSA',
        '$\log_{10}$(Population Estimate) per muSA',
        'Coverage Estimate per muSA'
    ],
    x_labels=['Inner-muSA Connections', 'Outgoing muSA Connections', 'Total muSA Connections',
              'muSA User Estimates', 'muSA Population Estimates', 'muSA Coverage Estimates'],
    no_log_cols=['coverage est'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\connectivity\\connections\\musa_connections_histograms.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# Connectivity per-capita and per-user histograms.
# ----------------------------------------------------------------------

# Per-capita / per-user columns at the county level.
df_inner_county['inter_county_connections per capita'] = df_inner_county['inter_county_connections'] / df_inner_county['pop_est']
df_inner_county['inter_county_connections per user'] = df_inner_county['inter_county_connections'] / df_inner_county['user_est']

df_inner_county['outer_county_connections per capita'] = df_inner_county['outer_county_connections'] / df_inner_county['pop_est']
df_inner_county['outer_county_connections per user'] = df_inner_county['outer_county_connections'] / df_inner_county['user_est']

df_inner_county['total connections per capita'] = df_inner_county['total connections'] / df_inner_county['pop_est']
df_inner_county['total connections per user'] = df_inner_county['total connections'] / df_inner_county['user_est']

# Per-capita / per-user columns at the CBSA level.
df_cbsa['inter_cbsa_connections per capita'] = df_cbsa['total inter_cbsa connections'] / df_cbsa['pop_est']
df_cbsa['inter_cbsa_connections per user'] = df_cbsa['total inter_cbsa connections'] / df_cbsa['user_est']

df_cbsa['outer_cbsa_connections per capita'] = df_cbsa['outer_cbsa_connections'] / df_cbsa['pop_est']
df_cbsa['outer_cbsa_connections per user'] = df_cbsa['outer_cbsa_connections'] / df_cbsa['user_est']

df_cbsa['total connections per capita'] = df_cbsa['total connections'] / df_cbsa['pop_est']
df_cbsa['total connections per user'] = df_cbsa['total connections'] / df_cbsa['user_est']

# Per-capita / per-user columns at the MSA level.
df_msa['inter_cbsa_connections per capita'] = df_msa['total inter_cbsa connections'] / df_msa['pop_est']
df_msa['inter_cbsa_connections per user'] = df_msa['total inter_cbsa connections'] / df_msa['user_est']

df_msa['outer_cbsa_connections per capita'] = df_msa['outer_cbsa_connections'] / df_msa['pop_est']
df_msa['outer_cbsa_connections per user'] = df_msa['outer_cbsa_connections'] / df_msa['user_est']

df_msa['total connections per capita'] = df_msa['total connections'] / df_msa['pop_est']
df_msa['total connections per user'] = df_msa['total connections'] / df_msa['user_est']

# Per-capita / per-user columns at the muSA level.
df_musa['inter_cbsa_connections per capita'] = df_musa['total inter_cbsa connections'] / df_musa['pop_est']
df_musa['inter_cbsa_connections per user'] = df_musa['total inter_cbsa connections'] / df_musa['user_est']

df_musa['outer_cbsa_connections per capita'] = df_musa['outer_cbsa_connections'] / df_musa['pop_est']
df_musa['outer_cbsa_connections per user'] = df_musa['outer_cbsa_connections'] / df_musa['user_est']

df_musa['total connections per capita'] = df_musa['total connections'] / df_musa['pop_est']
df_musa['total connections per user'] = df_musa['total connections'] / df_musa['user_est']

# County-level per-capita and per-user histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_inner_county] * 6,
    cols=['inter_county_connections per capita', 'outer_county_connections per capita', 'total connections per capita',
          'inter_county_connections per user', 'outer_county_connections per user', 'total connections per user'],
    titles=[
        '$\log_{10}$(Inner-County Connections per Capita) per County',
        '$\log_{10}$(Outgoing County Connections per Capita) per County',
        '$\log_{10}$(Total County Connections per Capita) per County',
        '$\log_{10}$(Inner-County Connections per User) per County',
        '$\log_{10}$(Outgoing County Connections per User) per County',
        '$\log_{10}$(Total County Connections per User) per County',
    ],
    x_labels=['Inner-County Connections per Capita', 'Outgoing County Connections per Capita', 'Total County Connections per Capita',
              'Inner-County Connections per User', 'Outgoing County Connections per User', 'Total County Connections per User'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\per_capita & per_user\\county_connections_histograms.png", dpi=800, bbox_inches='tight')

# CBSA-level per-capita and per-user histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['inter_cbsa_connections per capita', 'outer_cbsa_connections per capita', 'total connections per capita',
          'inter_cbsa_connections per user', 'outer_cbsa_connections per user', 'total connections per user'],
    titles=[
        '$\log_{10}$(Inner-CBSA Connections per Capita) per CBSA',
        '$\log_{10}$(Outgoing CBSA Connections per Capita) per CBSA',
        '$\log_{10}$(Total CBSA Connections per Capita) per CBSA',
        '$\log_{10}$(Inner-CBSA Connections per User) per CBSA',
        '$\log_{10}$(Outgoing CBSA Connections per User) per CBSA',
        '$\log_{10}$(Total CBSA Connections per User) per CBSA',
    ],
    x_labels=['Inner-CBSA Connections per Capita', 'Outgoing CBSA Connections per Capita', 'Total CBSA Connections per Capita',
              'Inner-CBSA Connections per User', 'Outgoing CBSA Connections per User', 'Total CBSA Connections per User'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\per_capita & per_user\\cbsa_connections_histograms.png", dpi=800, bbox_inches='tight')

# MSA-only per-capita and per-user histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['inter_cbsa_connections per capita', 'outer_cbsa_connections per capita', 'total connections per capita',
          'inter_cbsa_connections per user', 'outer_cbsa_connections per user', 'total connections per user'],
    titles=[
        '$\log_{10}$(Inner-MSA Connections per Capita) per MSA',
        '$\log_{10}$(Outgoing MSA Connections per Capita) per MSA',
        '$\log_{10}$(Total MSA Connections per Capita) per MSA',
        '$\log_{10}$(Inner-MSA Connections per User) per MSA',
        '$\log_{10}$(Outgoing MSA Connections per User) per MSA',
        '$\log_{10}$(Total MSA Connections per User) per MSA',
    ],
    x_labels=['Inner-MSA Connections per Capita', 'Outgoing MSA Connections per Capita', 'Total MSA Connections per Capita',
              'Inner-MSA Connections per User', 'Outgoing MSA Connections per User', 'Total MSA Connections per User'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\per_capita & per_user\\msa_connections_histograms.png", dpi=800, bbox_inches='tight')

# muSA-only per-capita and per-user histograms.
fig, axes = plot_histograms_with_pdfs_kde(
    dfs=[df_cbsa] * 6,
    cols=['inter_cbsa_connections per capita', 'outer_cbsa_connections per capita', 'total connections per capita',
          'inter_cbsa_connections per user', 'outer_cbsa_connections per user', 'total connections per user'],
    titles=[
        '$\log_{10}$(Inner-muSA Connections per Capita) per muSA',
        '$\log_{10}$(Outgoing muSA Connections per Capita) per muSA',
        '$\log_{10}$(Total muSA Connections per Capita) per muSA',
        '$\log_{10}$(Inner-muSA Connections per User) per muSA',
        '$\log_{10}$(Outgoing muSA Connections per User) per muSA',
        '$\log_{10}$(Total muSA Connections per User) per muSA',
    ],
    x_labels=['Inner-muSA Connections per Capita', 'Outgoing muSA Connections per Capita', 'Total muSA Connections per Capita',
              'Inner-muSA Connections per User', 'Outgoing muSA Connections per User', 'Total muSA Connections per User'],
    log_x=True,
    figsize_per_plot=5,
    bar_padding=0.02,
    title_fontsize=16,
    stats_fontsize=8,
    legend_fontsize=9
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\histograms\\per_capita & per_user\\musa_connections_histograms.png", dpi=800, bbox_inches='tight')

### Scaling Exponent Estimation and log-log Regression plots:

This is the headline analysis of the notebook. For each region type (County, CBSA, MSA, muSA) and each network type (Inter / Outgoing / Total), we:

1. Drop any rows where the regressor or response is non-positive (log is undefined there).
2. Log-transform both the normalized rescaled cumulative degree $K_r / \langle K_r \rangle$ and the normalized population $N / \langle N \rangle$.
3. Fit an OLS line using `statsmodels`, reporting $\beta$ (slope), $\gamma$ (intercept), $R^2$, RMSE, RSE, and the 95% CIs on both parameters.
4. Overlay the mean, median, and modal data points to give a visual sense of where the bulk of the GEOID distribution sits along the fit.

The slope $\beta$ is the quantity we care about — its position relative to 1 tells us whether social connectivity scales superlinearly, linearly, or sublinearly with population. The 95% CI on $\beta$ tells us whether the difference from 1 is statistically meaningful.

In [ ]:
"""
regression.py
=============
Fit and plot log-log OLS regressions of normalized rescaled cumulative
degree versus normalized population at the County, CBSA, MSA, and muSA
levels, for each of the three network types (Inter, Outgoing, Total).

The slope of each fit is the scaling exponent ``beta`` in the power-law
relation ``k = N^beta * eps``:

- ``beta > 1`` indicates superlinear scaling (bigger places generate
  disproportionately more connections per person).
- ``beta = 1`` indicates linear scaling (constant per-capita rate).
- ``beta < 1`` indicates sublinear scaling (proportionally fewer
  connections per person as population grows).

For each fit, the figure overlays the OLS line, a 95% confidence band
on the prediction, the mean / median / modal data points, and an inset
text box reporting ``beta``, ``gamma``, ``R^2``, RMSE, RSE, the 95%
confidence intervals on the slope and intercept, sample size, and the
mean ESRI user / population estimates.

Outputs (written to ``plots/regressions/``):

- ``county_connection_regressions.png`` (3 panels, one per network type)
- ``cbsa_connection_regressions.png``
- ``msa_connection_regressions.png``
- ``musa_connection_regressions.png``
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import geopandas as gpd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

# Load the four cleaned, rescaled, normalized dataframes produced by
# data_cleaning.py.
df_inner_county = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_outer_county.csv')
df_cbsa = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_cbsa.csv')
df_msa = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_msa.csv')
df_musa = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\export\\df_musa.csv')


def log_log_regression_plot_on_ax(df, x_col, y_col, ax, title=None, region=''):
    """Fit a log-log OLS regression and draw it on the given matplotlib axis.

    Strips non-positive rows (since log10 is undefined there), fits an
    OLS line in log10 space, and overlays the regression line, a 95%
    prediction confidence band, the data points, and reference markers
    for the mean, median, and modal point.

    Parameters
    ----------
    df : pandas.DataFrame
        Source dataframe; must contain ``x_col``, ``y_col``,
        ``user_est``, and ``pop_est`` columns.
    x_col, y_col : str
        Column names for the regressor and response. These are passed
        through log10 before fitting.
    ax : matplotlib.axes.Axes
        Axis to draw on.
    title : str, optional
        Subplot title.
    region : str, default ''
        Region label used in the axis labels (e.g. 'County', 'CBSA').

    Returns
    -------
    None
        Draws on ``ax`` in place. Use the inset legend text to read
        ``beta``, ``gamma``, ``R^2``, RMSE, RSE, the CIs, sample size,
        and the average user / population estimates.

    Raises
    ------
    ValueError
        If fewer than 2 positive rows remain after filtering.
    """
    df_plot = df[[x_col, y_col, 'user_est', 'pop_est']].copy()
    df_plot = df_plot[(df_plot[x_col] > 0) & (df_plot[y_col] > 0)]
    N = len(df_plot)

    if N < 2:
        raise ValueError("Not enough positive entries to fit regression.")

    # Log-transform both columns and fit the OLS model.
    x = np.log10(df_plot[x_col].values)
    y = np.log10(df_plot[y_col].values)

    x_with_const = sm.add_constant(x)
    model = sm.OLS(y, x_with_const)
    results = model.fit()

    slope = results.params[1]
    intercept = results.params[0]
    r2 = results.rsquared

    # 95% confidence intervals via the t-distribution with N-2 d.o.f.
    se = results.bse
    t_val = stats.t.ppf(0.975, df=N - 2)

    slope_ci = slope + np.array([-1, 1]) * t_val * se[1]
    intercept_ci = intercept + np.array([-1, 1]) * t_val * se[0]

    # Smooth grid for the fitted line and its prediction confidence band.
    x_fit = np.linspace(x.min(), x.max(), 200)
    y_fit = intercept + slope * x_fit

    y_pred_fit = results.get_prediction(sm.add_constant(x_fit))
    y_ci = y_pred_fit.conf_int(alpha=0.05)

    y_hat = results.predict(x_with_const)
    rmse = np.sqrt(mean_squared_error(y, y_hat))
    rse = np.sqrt(np.sum((y - y_hat) ** 2) / (N - 2))

    # Summary statistics of the data in log10 space.
    mean_x, mean_y = np.mean(x), np.mean(y)
    median_x, median_y = np.median(x), np.median(y)
    var_x, var_y = np.var(x, ddof=1), np.var(y, ddof=1)

    # Approximate the mode of the x-distribution by the densest histogram
    # bin; pair with the regression line to get a (mode_x, mode_y) point.
    hist, bin_edges = np.histogram(x, bins='auto')
    mode_bin = np.argmax(hist)
    mode_x = (bin_edges[mode_bin] + bin_edges[mode_bin + 1]) / 2
    mode_y = intercept + slope * mode_x

    # ---- Data points and reference markers (back in original units) ----
    ax.scatter(10**x, 10**y, color='darkblue', alpha=0.6, label='Data points')
    ax.scatter(10**mean_x, 10**mean_y, color='fuchsia', s=80, label='Mean')
    ax.scatter(10**median_x, 10**median_y, color='orange', s=80, label='Median')
    ax.scatter(10**mode_x, 10**mode_y, color='lime', s=80, label='Mode (approx.)')

    # ---- Regression line and confidence band ----
    ax.plot(10**x_fit, 10**y_fit, color='red', lw=2, label='Regression line')
    ax.fill_between(
        10**x_fit,
        10**y_ci[:, 0],
        10**y_ci[:, 1],
        color='darkred',
        alpha=0.2,
        label='95% Conf. Int.'
    )

    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.set_xlabel(
        rf'$\log_{{10}}\!\left(\frac{{N_i}}{{\langle N_{{\mathrm{{{region}}}}}\rangle}}\right)$ (Normed Pop. by {region})',
        fontsize=18
    )
    ax.set_ylabel(
        rf'$\log_{{10}}\!\left(\frac{{K_{{r,\mathrm{{total}}}}}}{{\langle K_{{r,\mathrm{{{region}}}}}\rangle}}\right)$ (Normed & Rescaled by {region})',
        fontsize=18
    )

    if title:
        ax.set_title(title, fontsize=24)

    ax.tick_params(axis='both', labelsize=10)

    # ---- Inset stats box ----
    legend_text = (
        f"beta (slope): {slope:.3f} [{slope_ci[0]:.3f}, {slope_ci[1]:.3f}]\n"
        f"gamma (intercept): {intercept:.3f} [{intercept_ci[0]:.3f}, {intercept_ci[1]:.3f}]\n"
        f"R^2: {r2:.3f}\n"
        f"RMSE: {rmse:.3f}\n"
        f"RSE: {rse:.3f}\n"
        f"Average User Est.: {df_plot['user_est'].mean():,.0f}\n"
        f"Average Pop Est.: {df_plot['pop_est'].mean():,.0f}\n"
        f"# of GEOIDs: {N}"
    )

    ax.text(
        0.05, 0.95, legend_text,
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='whitesmoke', alpha=0.8)
    )

    ax.legend(fontsize=12, loc='lower right')


# ----------------------------------------------------------------------
# County-level regressions: Inter / Outgoing / Total connection types
# against normed population. 3,141 cleaned counties.
# ----------------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)

log_log_regression_plot_on_ax(
    df=df_inner_county,
    x_col='normed pop_est',
    y_col='rescaled inter_county_connections',
    ax=axs[0],
    title='Inter-County Connections by County',
    region='County'
)

log_log_regression_plot_on_ax(
    df=df_inner_county,
    x_col='normed pop_est',
    y_col='rescaled outer_county_connections',
    ax=axs[1],
    title='Outgoing Connections by County',
    region='County'
)

log_log_regression_plot_on_ax(
    df=df_inner_county,
    x_col='normed pop_est',
    y_col='rescaled total connections',
    ax=axs[2],
    title='Total Connections by County',
    region='County'
)

plt.tight_layout()
plt.show()
# fig.savefig("F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\regressions\\county_connection_regressions.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# CBSA-level regressions (combined Metropolitan + Micropolitan). 917
# CBSAs.
# ----------------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)

log_log_regression_plot_on_ax(
    df=df_cbsa,
    x_col='normed pop_est',
    y_col='rescaled total inter_cbsa connections',
    ax=axs[0],
    title='Inter-County Connections by CBSA',
    region='CBSA'
)

log_log_regression_plot_on_ax(
    df=df_cbsa,
    x_col='normed pop_est',
    y_col='rescaled outer_cbsa_connections',
    ax=axs[1],
    title='Outgoing Connections by CBSA',
    region='CBSA'
)

log_log_regression_plot_on_ax(
    df=df_cbsa,
    x_col='normed pop_est',
    y_col='rescaled total connections',
    ax=axs[2],
    title='Total Connections by CBSA',
    region='CBSA'
)

plt.tight_layout()
plt.show()
# fig.savefig("F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\regressions\\cbsa_connection_regressions.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# MSA-only regressions. Normalization averages were already recomputed
# within the MSA subpopulation in data_cleaning.py.
# ----------------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)

log_log_regression_plot_on_ax(
    df=df_msa,
    x_col='normed pop_est',
    y_col='rescaled total inter_cbsa connections',
    ax=axs[0],
    title='Inter-County Connections by MSA',
    region='MSA'
)

log_log_regression_plot_on_ax(
    df=df_msa,
    x_col='normed pop_est',
    y_col='rescaled outer_cbsa_connections',
    ax=axs[1],
    title='Outgoing Connections by MSA',
    region='MSA'
)

log_log_regression_plot_on_ax(
    df=df_msa,
    x_col='normed pop_est',
    y_col='rescaled total connections',
    ax=axs[2],
    title='Total Connections by MSA',
    region='MSA'
)

plt.tight_layout()
plt.show()
# fig.savefig("F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\regressions\\msa_connection_regressions.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# muSA-only regressions. Same as MSA fits, but for the 536 Micropolitan
# Statistical Areas; CIs are wider because the population range is
# narrower.
# ----------------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)

log_log_regression_plot_on_ax(
    df=df_musa,
    x_col='normed pop_est',
    y_col='rescaled total inter_cbsa connections',
    ax=axs[0],
    title='Inter-County Connections by muSA',
    region='muSA'
)

log_log_regression_plot_on_ax(
    df=df_musa,
    x_col='normed pop_est',
    y_col='rescaled outer_cbsa_connections',
    ax=axs[1],
    title='Outgoing Connections by muSA',
    region='muSA'
)

log_log_regression_plot_on_ax(
    df=df_musa,
    x_col='normed pop_est',
    y_col='rescaled total connections',
    ax=axs[2],
    title='Total Connections by muSA',
    region='muSA'
)

plt.tight_layout()
plt.show()
# fig.savefig("F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\regressions\\musa_connection_regressions.png", dpi=800, bbox_inches='tight')

## Choropleth Plots:

Choropleths let us check that the connection counts have the *geographic* structure we expect — large metros lit up, sparse rural counties dim, coastal corridors visible — before we collapse everything to a single regression slope. We produce four sets of choropleths:

1. **Raw connection counts** — Inter-County, Outgoing County, Total County (log-color-scaled, because the counts span ~6 decades).
2. **Population-related statistics** — User estimates $|S_i|$, population estimates $N_i$, and coverage $s_i$.
3. **Per-capita connections** — connections divided by population, on a linear color scale.
4. **Per-user connections** — connections divided by Facebook user count, also linear.

The per-capita and per-user views are the most diagnostic: if the rescaling step in our regressions has done its job, the per-user choropleth should look noticeably *flatter* than the raw-count choropleth, because we've removed the population-scaling baseline.

In [ ]:
"""
choropleth.py
=============
Render county- and CBSA-level choropleth maps for the social-connection
variables produced by ``data_cleaning.py``.

Outputs (written to ``plots/choropleths/``):

- ``county_connections_choropleth.png``: Inter-, Outgoing, and Total
  county connections on a log color scale.
- ``county_popstats_choropleth.png``: County user-count estimates,
  population estimates (both log), and coverage estimates (linear).
- ``county_percapita_choropleth.png``: Connections per capita (linear).
- ``county_peruser_choropleth.png``: Connections per Facebook MAU
  (linear).

Maps are restricted to the continental US (state FIPS in 01-56,
excluding 02 Alaska, 15 Hawaii, and the non-state territories) so the
default extent is a sensible CONUS view.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.ticker as mticker
from esda.moran import Moran
from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, LogFormatter, ScalarFormatter
from matplotlib.colors import Normalize, LogNorm

# ----------------------------------------------------------------------
# Load the cleaned county- and CBSA-level dataframes alongside the 2021
# TIGER/Line shapefiles for the choropleth basemaps.
# ----------------------------------------------------------------------
df_inner_county = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_outer_county.csv')
df_cbsa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_cbsa.csv')
df_msa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_msa.csv')
df_musa = pd.read_csv('F:\\dsl_CLIMA\\submittable\\export\\df_musa.csv')
gdf_county = gpd.read_file('F:\\dsl_CLIMA\\submittable\\source\\shape files\\county\\tl_2021_us_county.shp')
gdf_cbsa = gpd.read_file('F:\\dsl_CLIMA\\submittable\\source\\shape files\\cbsa\\tl_2021_us_cbsa.shp')

# Force zero-padded string IDs so the dataframe joins do not silently
# drop leading-zero FIPS / CBSA codes.
df_inner_county['user_loc'] = df_inner_county['user_loc'].astype(str).str.zfill(5)
df_cbsa['CBSA Code'] = df_cbsa['CBSA Code'].astype(str).str.zfill(5)

gdf_county['user_loc'] = gdf_county['STATEFP'].str.zfill(2) + gdf_county['COUNTYFP'].str.zfill(3)
gdf_cbsa.rename(columns={'GEOID': 'CBSA Code'}, inplace=True)
gdf_cbsa['CBSA Code'] = gdf_cbsa['CBSA Code'].astype(str).str.zfill(5)

# Merge the variables of interest onto the polygon geodataframes.
gdf_county = gdf_county.merge(df_inner_county, on='user_loc', how='left')
gdf_county = gpd.GeoDataFrame(gdf_county, geometry='geometry', crs='EPSG:4326')

gdf_cbsa = gdf_cbsa.merge(df_cbsa, on='CBSA Code', how='left')
gdf_cbsa = gpd.GeoDataFrame(gdf_cbsa, geometry='geometry', crs='EPSG:4326')


def plot_choropleth_per_subplot(
    gdf,
    value_cols,
    titles=None,
    log_scale_cols=None,
    force_log_cols=None,
    cmap="viridis",
    missing_color="lightgrey",
    edgecolor="white",
    linewidth=0.2,
    figsize_per_plot=5,
    continental_only=True,
    zoom_padding=0.05,
    cbar_height=0.025,
    cbar_pad=0.012,
    cbar_width_ratio=0.75,
    linear_ticks=6,
    log_ticks_per_decade=1
):
    """Render a grid of choropleth subplots from a single GeoDataFrame.

    Each subplot maps one column from ``value_cols`` and gets its own
    horizontal colorbar positioned beneath the panel. Columns named in
    ``log_scale_cols`` are log-transformed before plotting (the colorbar
    ticks still display the original units). Columns named in
    ``force_log_cols`` are plotted on a linear scale but receive a
    log-spaced colorbar; this is useful when the underlying data has
    been pre-rescaled but a log-style colorbar reads better.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Source geodataframe; must contain a ``geometry`` column and a
        ``STATEFP`` or ``GEOID`` column if ``continental_only=True``.
    value_cols : list of str
        Column names to map, in subplot order.
    titles : list of str, optional
        Per-subplot titles. Defaults to ``value_cols``.
    log_scale_cols : list of str, optional
        Columns whose values are log10-transformed before plotting.
    force_log_cols : list of str, optional
        Columns whose colorbars get log-style ticks regardless of
        whether the underlying data was log-transformed.
    cmap : str, default "viridis"
        Matplotlib colormap.
    missing_color : str, default "lightgrey"
        Color for GEOIDs with missing values.
    edgecolor, linewidth : str, float
        Polygon outline styling.
    figsize_per_plot : float
        Size in inches allocated to each subplot.
    continental_only : bool, default True
        If True, filter to CONUS state FIPS only.
    zoom_padding : float, default 0.05
        Fraction of total bounds shaved off each edge to zoom in.
    cbar_height, cbar_pad, cbar_width_ratio : float
        Colorbar geometry parameters.
    linear_ticks : int, default 6
        Number of ticks on linear colorbars.
    log_ticks_per_decade : float
        Density of ticks on log colorbars (1 = one per decade).

    Returns
    -------
    fig : matplotlib.figure.Figure
    axes : ndarray of matplotlib.axes.Axes
    """
    if titles is None:
        titles = value_cols
    if log_scale_cols is None:
        log_scale_cols = []
    if force_log_cols is None:
        force_log_cols = []

    gdf_plot = gdf.copy()

    # ---- Continental US filter ----
    if continental_only:
        continental_fips = [str(i).zfill(2) for i in range(1, 57)
                            if i not in [2, 15, 60, 66, 69, 72, 78]]
        if "STATEFP" in gdf_plot.columns:
            gdf_plot = gdf_plot[gdf_plot["STATEFP"].isin(continental_fips)]
        elif "GEOID" in gdf_plot.columns:
            gdf_plot = gdf_plot[gdf_plot["GEOID"].str[:2].isin(continental_fips)]
        else:
            raise ValueError("Cannot filter continental US: no STATEFP or GEOID column found.")

    n = len(value_cols)
    cols = min(3, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(cols * figsize_per_plot, rows * figsize_per_plot),
        constrained_layout=False
    )
    axes = axes.flatten()

    fig.subplots_adjust(
        left=0.02,
        right=0.98,
        top=0.95,
        bottom=0.08,
        hspace=0.10,
        wspace=0.05
    )

    # ---- Zoom bounds with padding ----
    minx, miny, maxx, maxy = gdf_plot.total_bounds
    dx = (maxx - minx) * zoom_padding
    dy = (maxy - miny) * zoom_padding
    bounds = (minx + dx, miny + dy, maxx - dx, maxy - dy)

    for ax, col, title in zip(axes, value_cols, titles):

        data = gdf_plot[col].replace(0, np.nan)
        vmin, vmax = data.min(), data.max()

        # Log-transform data if requested; otherwise plot linearly.
        # ``is_log`` controls only the colorbar tick formatting.
        if col in log_scale_cols:
            plot_data = np.where(data > 0, np.log10(data), np.nan)
            norm = Normalize(vmin=np.nanmin(plot_data), vmax=np.nanmax(plot_data))
            is_log = True
        else:
            plot_data = data
            norm = Normalize(vmin=vmin, vmax=vmax)
            is_log = col in force_log_cols

        gdf_plot.plot(
            column=plot_data,
            ax=ax,
            cmap=cmap,
            norm=norm,
            missing_kwds={"color": missing_color},
            edgecolor=edgecolor,
            linewidth=linewidth
        )

        ax.set_title(title, fontsize=12)
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        ax.axis("off")

        # ---- Colorbar positioned just below each subplot ----
        pos = ax.get_position()
        width = pos.width * cbar_width_ratio
        x0 = pos.x0 + (pos.width - width) / 2
        cax = fig.add_axes([x0, pos.y0 - cbar_pad - cbar_height, width, cbar_height])

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        cbar = plt.colorbar(sm, cax=cax, orientation="horizontal")
        cbar.ax.xaxis.get_offset_text().set_visible(False)
        cbar.ax.set_xlabel("")
        cbar.ax.tick_params(labelsize=9)

        # ---- Tick formatting ----
        if col in log_scale_cols:
            # Data is log-transformed; show original units on the ticks.
            log_ticks = np.logspace(np.floor(np.log10(vmin)),
                                    np.ceil(np.log10(vmax)),
                                    int((np.ceil(np.log10(vmax)) - np.floor(np.log10(vmin))) * log_ticks_per_decade + 1))
            log_ticks = log_ticks[(log_ticks >= vmin) & (log_ticks <= vmax)]
            labels = [f"{int(t):,}" for t in log_ticks]
            cbar.ax.xaxis.set_major_locator(FixedLocator(np.log10(log_ticks)))
            cbar.ax.xaxis.set_major_formatter(FixedFormatter(labels))

        elif col in force_log_cols:
            # Linear data, but the colorbar is forced onto a log scale
            # so the ticks read as 1, 10, 100, etc. in original units.
            log_ticks = np.logspace(np.floor(np.log10(vmin)),
                                    np.ceil(np.log10(vmax)),
                                    int((np.ceil(np.log10(vmax)) - np.floor(np.log10(vmin))) * log_ticks_per_decade + 1))
            log_ticks = log_ticks[(log_ticks >= vmin) & (log_ticks <= vmax)]
            log_norm_ticks = (np.log10(log_ticks) - np.log10(vmin)) / (np.log10(vmax) - np.log10(vmin))
            labels = [f"{int(t):,}" for t in log_ticks]
            cbar.ax.xaxis.set_major_locator(FixedLocator(log_norm_ticks * (cbar.ax.get_xlim()[1] - cbar.ax.get_xlim()[0]) + cbar.ax.get_xlim()[0]))
            cbar.ax.xaxis.set_major_formatter(FixedFormatter(labels))

        else:
            # Linear data, linear colorbar.
            ticks = np.linspace(vmin, vmax, linear_ticks)
            labels = [f"{t:.2f}" if t < 10 else f"{int(t)}" for t in ticks]
            cbar.ax.xaxis.set_major_locator(FixedLocator(ticks))
            cbar.ax.xaxis.set_major_formatter(FixedFormatter(labels))

    # Hide unused axes when ``n`` doesn't fill the grid.
    for ax in axes[n:]:
        ax.axis("off")

    return fig, axes


# ----------------------------------------------------------------------
# County-level connection choropleths (log color scale).
# These show absolute Inter-, Outgoing, and Total connection counts.
# ----------------------------------------------------------------------
fig, axes = plot_choropleth_per_subplot(
    gdf=gdf_county,
    value_cols=[
        "inter_county_connections",
        "outer_county_connections",
        "total connections"
    ],
    titles=[
        "Inter-County Connections",
        "Outgoing County Connections",
        "Total County Connections"
    ],
    log_scale_cols=['inter_county_connections', 'outer_county_connections', 'total connections'],
    cmap="viridis",
    continental_only=True,
    zoom_padding=-0.075,
    log_ticks_per_decade=0.5,
    cbar_height=0.01,
    cbar_pad=0.012
)

plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\choropleths\\county_connections_choropleth.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# User-count, population, and coverage choropleths.
# User counts and population are log-scaled because they span ~6
# decades; coverage is linear because it sits in [0.34, 0.63].
# ----------------------------------------------------------------------
fig, axes = plot_choropleth_per_subplot(
    gdf=gdf_county,
    value_cols=[
        "user_est",
        "pop_est",
        "coverage est"
    ],
    titles=[
        "County User Count Estimates",
        "County Population Estimates",
        "County Coverage Estimates"
    ],
    log_scale_cols=['user_est', 'pop_est'],
    force_log_cols=['user_est', 'pop_est'],
    cmap="viridis",
    continental_only=True,
    zoom_padding=-0.075,
    log_ticks_per_decade=0.5,
    cbar_height=0.01,
    cbar_pad=0.012,
    linear_ticks=5
)

plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\choropleths\\county_popstats_choropleth.png", dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# Per-capita and per-user connection columns.
# Computed in-place on ``gdf_county`` for both the percapita and peruser
# choropleths below.
# ----------------------------------------------------------------------
gdf_county['inter_county_connections per capita'] = gdf_county['inter_county_connections'] / gdf_county['pop_est']
gdf_county['inter_county_connections per user'] = gdf_county['inter_county_connections'] / gdf_county['user_est']

gdf_county['outer_county_connections per capita'] = gdf_county['outer_county_connections'] / gdf_county['pop_est']
gdf_county['outer_county_connections per user'] = gdf_county['outer_county_connections'] / gdf_county['user_est']

gdf_county['total connections per capita'] = gdf_county['total connections'] / gdf_county['pop_est']
gdf_county['total connections per user'] = gdf_county['total connections'] / gdf_county['user_est']


# ----------------------------------------------------------------------
# Per-capita choropleths. Linear scale because the per-capita normalizer
# already strips out the population-driven baseline.
# ----------------------------------------------------------------------
fig, axes = plot_choropleth_per_subplot(
    gdf=gdf_county,
    value_cols=[
        "inter_county_connections per capita",
        "outer_county_connections per capita",
        "total connections per capita"
    ],
    titles=[
        "Inter-County Connections per Capita",
        "Outgoing County Connections per Capita",
        "Total County Connections per Capita"
    ],
    cmap="viridis",
    continental_only=True,
    zoom_padding=-0.075,
    log_ticks_per_decade=0.5,
    cbar_height=0.01,
    cbar_pad=0.012,
    linear_ticks=5
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\choropleths\\county_percapita_choropleth.png", dpi=800, bbox_inches='tight')


# ----------------------------------------------------------------------
# Per-user choropleths. This is the post-coverage-rescaling view: any
# remaining geographic structure here is not driven by population size.
# ----------------------------------------------------------------------
fig, axes = plot_choropleth_per_subplot(
    gdf=gdf_county,
    value_cols=[
        "inter_county_connections per user",
        "outer_county_connections per user",
        "total connections per user"
    ],
    titles=[
        "Inner-County Connections per User",
        "Outgoing County Connections per User",
        "Total County Connections per User"
    ],
    cmap="viridis",
    continental_only=True,
    zoom_padding=-0.075,
    log_ticks_per_decade=0.5,
    cbar_height=0.01,
    cbar_pad=0.012,
    linear_ticks=5
)
plt.show()
fig.savefig("F:\\dsl_CLIMA\\submittable\\plots\\choropleths\\county_peruser_choropleth.png", dpi=800, bbox_inches='tight')

## Demographics Plots:

The demographics section zooms from the national scaling picture down to a handful of NYC census blocks — specifically, the Hamilton Beach and Howard Beach blocks in Queens (FIPS prefix `360810884006`), which are the focus communities for CLIMA's coastal-flood-risk interviews, with Red Hook (Brooklyn) included later as a comparison.

For each block (and the aggregate across all selected blocks), we generate:

- **Age/Sex population pyramids** with the NYC census-block average overlaid as dots, so each block can be read against the citywide baseline.
- **Race/Ethnicity distributions** (White non-Hispanic, Black non-Hispanic, Asian non-Hispanic, Hispanic of any race), again with citywide averages overlaid.
- **Household size distributions** (1-person through 5+-person, with 5+, 6, 7+ collapsed for readability).
- **Housing occupancy** (occupied vs. vacant).
- **Housing tenure** (owner- vs. renter-occupied) — the most policy-relevant variable for the homeowner-mobility framing of CLIMA.

The data source for everything in this cell is the 2020 Decennial Census DHC files at the block level (H3, H4, H9, P3, P5, P12).

In [ ]:
"""
demographics.py
===============
Census-block demographic figures for the Hamilton Beach + Howard Beach
focus communities (FIPS prefix ``360810884006``), with NYC-wide
census-block averages overlaid for comparison.

Data sources (read from ``source/census demographic files/``):

- ``DECENNIALDHC2020.H3-Data.csv``: housing occupancy counts (H3 table).
- ``DECENNIALDHC2020.H4-Data.csv``: housing tenure counts (H4 table).
- ``DECENNIALDHC2020.H9-Data.csv``: household size counts (H9 table).
- ``DECENNIALDHC2020.P3-Data.csv``: race counts (P3 table).
- ``DECENNIALDHC2020.P5-Data.csv``: Hispanic or Latino origin by race
  (P5 table).
- ``DECENNIALDHC2020.P12-Data.csv``: sex by age across the total
  population (P12 table).

For each target block, and for the aggregate over all target blocks,
the script produces:

- Age / sex population pyramids.
- Race / ethnicity distributions (White non-Hispanic, Black non-Hispanic,
  Asian non-Hispanic, Hispanic of any race).
- Household size distributions (1, 2, 3, 4, 5+ persons).
- Housing occupancy (occupied vs. vacant).
- Housing tenure (owner-occupied vs. renter-occupied).

Outputs are written under ``plots/demographics/bar charts/`` in
sub-folders by figure type.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Load every relevant 2020 Decennial Census DHC file. GEO_ID and NAME
# are read as strings so the 22-character block identifiers and full
# block names survive intact.
# ----------------------------------------------------------------------
df_tenure = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H4-Data.csv', dtype={'GEO_ID': str, 'NAME': str})
df_household_size = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H9-Data.csv', dtype={'GEO_ID': str, 'NAME': str})
df_sex_by_age = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P12-Data.csv', dtype={'GEO_ID': str, 'NAME': str})
df_race = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P3-Data.csv', dtype={'GEO_ID': str, 'NAME': str})
df_hispanic = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.P5-Data.csv', dtype={'GEO_ID': str, 'NAME': str})
df_occupancy = pd.read_csv('F:\\dsl_CLIMA\\projects\\submittable\\clima\\source\\census demographic files\\DECENNIALDHC2020.H3-Data.csv', dtype={'GEO_ID': str, 'NAME': str})

# Drop the duplicate ``NAME`` columns from every file but the first so
# the outer merge below doesn't create _x / _y collisions on NAME.
df_household_size = df_household_size.drop(columns=['NAME'])
df_sex_by_age = df_sex_by_age.drop(columns=['NAME'])
df_race = df_race.drop(columns=['NAME'])
df_hispanic = df_hispanic.drop(columns=['NAME'])
df_occupancy = df_occupancy.drop(columns=['NAME'])

# Outer-merge every table on GEO_ID so each block has all six DHC
# tables in one row. astype(str) at each step preserves the GEOID dtype
# through the merges.
df_merged = df_tenure.merge(df_household_size, on='GEO_ID', how='outer').astype(str) \
               .merge(df_sex_by_age, on='GEO_ID', how='outer').astype(str) \
               .merge(df_race, on='GEO_ID', how='outer').astype(str) \
               .merge(df_hispanic, on='GEO_ID', how='outer').astype(str) \
               .merge(df_occupancy, on='GEO_ID', how='outer').astype(str)
df_merged = df_merged.drop(columns=['Unnamed: 6', 'Unnamed: 10_x', 'Unnamed: 51', 'Unnamed: 10_y', 'Unnamed: 19', 'Unnamed: 5'])

# Target census blocks in Hamilton Beach / Howard Beach (Queens).
# All share the FIPS prefix 360810884006 (state=36 NY, county=081 Queens,
# tract=088400, block group=6).
target_locations = [
    '1000000US360810884006000', '1000000US360810884006001', '1000000US360810884006003',
    '1000000US360810884006005', '1000000US360810884006004', '1000000US360810884006006',
    '1000000US360810884006008', '1000000US360810884006009', '1000000US360810884006010'
]

df_filtered = df_merged.copy()
df_filtered = df_filtered[df_filtered['GEO_ID'].isin(target_locations)]

df_filtered.columns = df_filtered.columns.str.strip()
df_merged.columns = df_merged.columns.str.strip()


def plot_population_pyramid(df_subset, df_all):
    """Plot an aggregate age / sex pyramid for the target census blocks.

    Bars show the target-block percentage of total target-block
    population in each age bucket, split into male (left, blue) and
    female (right, pink). NYC-wide census-block averages are overlaid
    as dots for direct comparison.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks (used to compute the citywide
        averages).
    """
    # P12 male age columns: P12_003N..P12_025N (23 buckets).
    # P12 female age columns: P12_027N..P12_049N (23 buckets).
    male_cols = [f'P12_{i:03}N' for i in range(3, 26)]
    female_cols = [f'P12_{i:03}N' for i in range(27, 50)]

    age_labels = [
        'Under 5', '5-9', '10-14', '15-17', '18-19', '20', '21', '22-24',
        '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59',
        '60-61', '62-64', '65-66', '67-69', '70-74', '75-79', '80-84', '85+'
    ]
    y = np.arange(len(age_labels))

    # Coerce numeric. The merge step left everything as strings.
    cols_to_convert = male_cols + female_cols + ['P12_001N']
    df_all[cols_to_convert] = df_all[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_subset[cols_to_convert] = df_subset[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Target-block age percentages.
    male_totals = df_subset[male_cols].sum()
    female_totals = df_subset[female_cols].sum()
    total_subset = male_totals.sum() + female_totals.sum()
    male_percent = (male_totals / total_subset) * 100
    female_percent = (female_totals / total_subset) * 100

    # NYC-wide averages, restricting to non-empty blocks.
    df_nonzero = df_all[df_all['P12_001N'] > 0].copy()
    total_all = df_nonzero[male_cols + female_cols].sum().sum()
    male_avg = df_nonzero[male_cols].sum() / total_all * 100
    female_avg = df_nonzero[female_cols].sum() / total_all * 100

    max_male = max(male_percent.max(), male_avg.max()) * 1.1
    max_female = max(female_percent.max(), female_avg.max()) * 1.1

    fig, ax = plt.subplots(figsize=(14, 9))

    # First pass: render the age labels invisibly so we can measure
    # their width in data coordinates and reserve a center channel that
    # the bars won't overlap.
    txts = []
    for i, label in enumerate(age_labels):
        txt = ax.text(0, i, label, ha='center', va='center', fontsize=16, fontweight='bold',
                      bbox=dict(facecolor='white', edgecolor='none', alpha=0.9))
        txts.append(txt)

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    label_widths = []
    for txt in txts:
        bbox = txt.get_window_extent(renderer=renderer)
        inv = ax.transData.inverted()
        left_data = inv.transform((bbox.x0, 0))[0]
        right_data = inv.transform((bbox.x1, 0))[0]
        width_data = right_data - left_data
        label_widths.append(width_data)

    max_label_width = max(label_widths)
    center_padding = max_label_width + 0.5

    ax.cla()

    # Male bars extend left from -center_padding; female bars extend
    # right from +center_padding. The gap in between holds the age
    # labels.
    ax.barh(y, -male_percent.values, color='steelblue', alpha=0.7, label='Percent Male',
            left=-center_padding)
    ax.barh(y, female_percent.values, color='lightcoral', alpha=0.7, label='Percent Female',
            left=center_padding)

    # NYC-average reference dots, shifted out by the center padding so
    # they align with their respective sides.
    ax.plot(-male_avg.values - center_padding, y, 'o', color='navy', label='NYC Census Block Avg Male')
    ax.plot(female_avg.values + center_padding, y, 'o', color='darkred', label='NYC Census Block Avg Female')

    # Age labels rendered in the center channel.
    for i, label in enumerate(age_labels):
        ax.text(0, i, label, ha='center', va='center', fontsize=16, fontweight='bold',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.9))

    # Vertical baselines at the two true zero lines.
    ax.vlines(-center_padding, ymin=-1, ymax=len(age_labels), colors='black', linewidth=1.5)
    ax.vlines(center_padding, ymin=-1, ymax=len(age_labels), colors='black', linewidth=1.5)

    ax.set_xlim(-max_male - center_padding * 2, max_female + center_padding * 2)
    ax.set_ylim(-1, len(age_labels))

    # Mirrored, padding-adjusted x-axis ticks.
    left_ticks = np.linspace(0, max_male, 6)
    right_ticks = np.linspace(0, max_female, 6)
    ticks_left = -left_ticks - center_padding
    ticks_right = right_ticks + center_padding
    all_ticks = np.concatenate((ticks_left[::-1], [0], ticks_right))
    ax.set_xticks(all_ticks)

    tick_labels = [f"{int(abs(x + center_padding))}%" if x < 0 else "" for x in ticks_left[::-1]]
    tick_labels += [""]
    tick_labels += [f"{int(x - center_padding)}%" if x > 0 else "" for x in ticks_right]
    ax.set_xticklabels(tick_labels)

    ax.set_xlabel("Percentage of Total Population (%)", fontsize=16)
    ax.xaxis.set_ticks_position('bottom')
    ax.xaxis.set_label_position('bottom')

    ax.set_yticks([])

    ax.axvline(0, color='black', linewidth=1)

    ax.text(-max_male - center_padding, len(age_labels) + 0.5, "Male", ha='center',
            fontsize=12, fontweight='bold', color='steelblue')
    ax.text(max_female + center_padding, len(age_labels) + 0.5, "Female", ha='center',
            fontsize=12, fontweight='bold', color='darkred')

    ax.set_title("Age/Sex Population Distribution for Hamilton Beach Census Blocks", fontsize=16, fontweight='bold')
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()
    fig.savefig("F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\allBlocks\\hb_allBlocks_popPyramid.png", dpi=800, bbox_inches='tight')


plot_population_pyramid(df_filtered, df_merged)


def plot_population_pyramid_by_block(df_subset, df_all):
    """Same as ``plot_population_pyramid`` but emits one figure per block.

    Each block's percentages are normalized by that block's own total
    population, so the bars are directly comparable across blocks even
    when the absolute populations differ.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks (citywide averages).
    """
    male_cols = [f'P12_{i:03}N' for i in range(3, 26)]
    female_cols = [f'P12_{i:03}N' for i in range(27, 50)]

    age_labels = [
        'Under 5', '5-9', '10-14', '15-17', '18-19', '20', '21', '22-24',
        '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59',
        '60-61', '62-64', '65-66', '67-69', '70-74', '75-79', '80-84', '85+'
    ]
    y = np.arange(len(age_labels))

    label_fontsize = 16
    # Fixed center channel width; simpler than the dynamic measurement
    # in the aggregate version because per-block plots are smaller.
    center_pad = 2

    cols_to_convert = male_cols + female_cols + ['P12_001N']
    df_all[cols_to_convert] = df_all[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_subset[cols_to_convert] = df_subset[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)

    for idx, row in df_subset.iterrows():
        male_counts = row[male_cols]
        female_counts = row[female_cols]
        total_population = male_counts.sum() + female_counts.sum()
        if total_population == 0:
            print(f"Skipping block {idx} with zero population")
            continue

        male_percent = (male_counts / total_population) * 100
        female_percent = (female_counts / total_population) * 100

        df_nonzero = df_all[df_all['P12_001N'] > 0]
        total_all = df_nonzero[male_cols + female_cols].sum().sum()
        male_avg = df_nonzero[male_cols].sum() / total_all * 100
        female_avg = df_nonzero[female_cols].sum() / total_all * 100

        # Symmetric x-axis range rounded up to the nearest 5% so the
        # ticks are clean across blocks.
        max_pct = max(male_percent.max(), female_percent.max(), male_avg.max(), female_avg.max())
        max_pct = np.ceil(max_pct / 5.0) * 5

        fig, ax = plt.subplots(figsize=(14, 9))

        ax.axvline(-center_pad, color='black', linewidth=2.5, zorder=5)
        ax.axvline(center_pad, color='black', linewidth=2.5, zorder=5)

        ax.barh(y, -male_percent.values, left=-center_pad,
                color='steelblue', alpha=0.7, label='Percent Male', zorder=3)
        ax.barh(y, female_percent.values, left=center_pad,
                color='lightcoral', alpha=0.7, label='Percent Female', zorder=3)

        ax.plot(-male_avg.values - center_pad, y, 'o', color='navy', label='NYC Avg Male', zorder=4)
        ax.plot(female_avg.values + center_pad, y, 'o', color='darkred', label='NYC Avg Female', zorder=4)

        for i, label in enumerate(age_labels):
            ax.text(0, i, label, ha='center', va='center', fontsize=label_fontsize,
                    fontweight='bold', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9), zorder=2)

        ax.text(-center_pad - max_pct, len(age_labels) + 0.8, "Male",
                ha='center', fontsize=label_fontsize + 2, fontweight='bold', color='steelblue')
        ax.text(center_pad + max_pct, len(age_labels) + 0.8, "Female",
                ha='center', fontsize=label_fontsize + 2, fontweight='bold', color='darkred')

        ax.set_xlim(-center_pad - max_pct, center_pad + max_pct)
        ax.set_ylim(-1, len(age_labels))
        ax.set_yticks([])

        tick_vals = np.linspace(0, max_pct, 6)
        ticks_left = -tick_vals - center_pad
        ticks_right = tick_vals + center_pad
        ax.set_xticks(np.concatenate((ticks_left[::-1], [0], ticks_right)))
        ax.set_xticklabels(
            [f"{int(t)}%" for t in tick_vals[::-1]] + [""] + [f"{int(t)}%" for t in tick_vals],
            fontsize=label_fontsize
        )

        ax.set_xlabel("Percentage of Total Population (%)", fontsize=label_fontsize + 2)
        ax.xaxis.set_ticks_position('bottom')
        ax.xaxis.set_label_position('bottom')

        # Last four chars of the GEO_ID = block-within-block-group code.
        block_code = str(row['GEO_ID'])[-4:]
        ax.set_title(f"Age/Sex Population Distribution for Block {block_code}",
                     fontsize=label_fontsize + 4, fontweight='bold')

        ax.legend(loc='upper right', fontsize=label_fontsize)
        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\population pyramid\\hb_{block_code}_popPyramid.png", dpi=800, bbox_inches='tight')


plot_population_pyramid_by_block(df_filtered, df_merged)


def plot_race_distribution_by_block(df_subset, df_all):
    """Plot the race / ethnicity distribution for the target blocks.

    Renders one aggregate plot for the union of the target blocks and
    one plot per individual block. Categories are sorted by NYC-wide
    descending percentage so the order is consistent across panels.
    Hispanic / Latino is treated as a single combined category that
    cuts across race.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks.
    """
    # Selected race categories (non-Hispanic alone).
    race_cols = {
        "White alone, Non-Hispanic": "P5_003N",
        "Black or African American alone, Non-Hispanic": "P5_004N",
        "Asian alone, Non-Hispanic": "P5_006N",
    }
    # P5_011N..P5_017N covers Hispanic-or-Latino respondents across
    # multiple race classifications; we sum across them for a single
    # combined "Hispanic or Latino (All)" category.
    hispanic_cols = [f'P5_{i:03}N' for i in range(11, 18)]
    all_cols = list(race_cols.values()) + hispanic_cols + ['P5_001N']

    df_subset[all_cols] = df_subset[all_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_all[all_cols] = df_all[all_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    total_all_pop = df_all["P5_001N"].sum()
    race_vals_all = {label: df_all[col].sum() for label, col in race_cols.items()}
    race_vals_all["Hispanic or Latino (All)"] = df_all[hispanic_cols].sum().sum()
    avg_pct = {k: (v / total_all_pop) * 100 for k, v in race_vals_all.items()}

    # Sort categories by NYC-wide percentage so the largest groups
    # appear first.
    sorted_items = sorted(avg_pct.items(), key=lambda x: x[1], reverse=True)
    categories = [k for k, _ in sorted_items]

    # ---- Aggregate plot across all target blocks ----
    subset_total = df_subset[race_cols.values()].sum()
    subset_hispanic_total = df_subset[hispanic_cols].sum().sum()
    subset_pop_total = df_subset["P5_001N"].sum()

    if subset_pop_total > 0:
        combined_vals = {label: subset_total[race_cols[label]] for label in race_cols}
        combined_vals["Hispanic or Latino (All)"] = subset_hispanic_total
        combined_pct = {k: (v / subset_pop_total) * 100 for k, v in combined_vals.items()}

        print(f"\n--- Combined Race/Ethnicity Distribution for Hamilton/Howard Beach Census Blocks ---")
        for label in categories:
            count = int(combined_vals[label])
            print(f"  {label}: {count}")
        print(f"Total population: {int(subset_pop_total)}")

        values = [combined_pct[k] for k in categories]
        averages = [avg_pct[k] for k in categories]
        y = np.arange(len(categories))

        fig, ax = plt.subplots(figsize=(16, 6))
        ax.barh(y, values, color='steelblue', alpha=0.85, label='Hamilton/Howard Beach Census Blocks Combined')
        ax.plot(averages, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(categories, fontsize=16)
        ax.invert_yaxis()
        ax.set_xlabel('Percentage of Total Population (%)', fontsize=20)
        ax.tick_params(axis='x', labelsize=16)
        ax.set_title('Race/Ethnicity Distribution (Hamilton/Howard Beach Census Blocks)', fontsize=20, fontweight='bold')
        ax.legend(loc='lower right', fontsize=14)

        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\allBlocks\\hb_allBlocks_raceDistribution.png", dpi=800, bbox_inches='tight')

    # ---- Per-block plots ----
    for idx, row in df_subset.iterrows():
        geo_id = str(row.get('GEO_ID', f'{idx}'))
        block_name = geo_id[-4:]
        total_pop = row["P5_001N"]
        if total_pop == 0:
            print(f"Skipping {block_name} due to zero population.")
            continue

        race_vals = {label: row[race_cols[label]] for label in race_cols}
        race_vals["Hispanic or Latino (All)"] = row[hispanic_cols].sum()
        race_pct = {k: (v / total_pop) * 100 for k, v in race_vals.items()}

        print(f"\n--- Race/Ethnicity Distribution for {block_name} ---")
        for label in categories:
            count = int(race_vals[label])
            print(f"  {label}: {count}")
        print(f"Total population: {int(total_pop)}")

        values = [race_pct[k] for k in categories]
        averages = [avg_pct[k] for k in categories]
        y = np.arange(len(categories))

        fig, ax = plt.subplots(figsize=(16, 6))
        ax.barh(y, values, color='steelblue', alpha=0.85, label=block_name)
        ax.plot(averages, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(categories, fontsize=16)
        ax.invert_yaxis()
        ax.set_xlabel('Percentage of Total Population', fontsize=20)
        ax.tick_params(axis='x', labelsize=16)
        ax.set_title(f'Race/Ethnicity Distribution for {block_name}', fontsize=20, fontweight='bold')
        ax.legend(loc='lower right', fontsize=14)

        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\race distribution\\hb_{block_name}_raceDistribution.png", dpi=800, bbox_inches='tight')


plot_race_distribution_by_block(df_filtered, df_merged)


def plot_household_size_distribution(df_subset, df_all):
    """Plot household-size distribution (1, 2, 3, 4, 5+ persons).

    The raw H9 table breaks 5+-person households into three buckets
    (5, 6, 7+). For readability, we collapse these into a single 5+
    category. One aggregate plot is produced across target blocks plus
    one per individual block.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks.
    """
    # Raw H9 size columns (H9_002N..H9_008N: 1-person through 7+).
    original_cols = [f'H9_{i:03}N' for i in range(2, 9)]
    total_col = 'H9_001N'
    cols_to_convert = original_cols + [total_col]

    df_all[cols_to_convert] = df_all[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_subset[cols_to_convert] = df_subset[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Collapse 5/6/7+ into a single 5+ category in both dataframes.
    for df in [df_all, df_subset]:
        df['H9_006N'] = df[['H9_006N', 'H9_007N', 'H9_008N']].sum(axis=1)

    size_cols = [f'H9_{i:03}N' for i in range(2, 7)]
    labels = [
        '1-person household',
        '2-person household',
        '3-person household',
        '4-person household',
        '5-or-more-person household'
    ]

    total_all = df_all[total_col].sum()
    all_counts = df_all[size_cols].sum()
    pct_all = (all_counts / total_all * 100).values

    subset_total = df_subset[total_col].sum()
    subset_counts = df_subset[size_cols].sum()
    subset_pct = (subset_counts / subset_total * 100).values

    # Aggregate across target blocks.
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(labels, subset_pct, color='seagreen', label='Hamilton/Howard Beach Census Blocks Total')
    ax.plot(pct_all, labels, 'ko', label='NYC Census Block Avg')

    ax.set_xlabel('Percentage of Households', fontsize=12)
    ax.set_title('Household Size Distribution (1-5+ persons)\nHamilton/Howard Beach Census Blocks', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.legend()
    plt.tight_layout()
    plt.show()
    fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\allBlocks\\hb_allBlocks_householdsizeDistribution.png", dpi=800, bbox_inches='tight')

    # Per-block.
    for _, row in df_subset.iterrows():
        geo_id = str(row['GEO_ID'])
        block_id = geo_id[-4:]
        block_counts = row[size_cols].astype(float)
        total = row[total_col]

        if total == 0:
            continue

        pct = (block_counts / total * 100).values

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(labels, pct, color='mediumseagreen', label=f'Block {block_id}')
        ax.plot(pct_all, labels, 'ko', label='NYC Census Block Avg')

        ax.set_xlabel('Percentage of Households', fontsize=12)
        ax.set_title(f'Household Size Distribution (1-5+ persons)\nBlock {block_id}', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
    fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\race distribution\\hb_{block_id}_householdsizeDistribution.png", dpi=800, bbox_inches='tight')


plot_household_size_distribution(df_filtered, df_merged)


def plot_occupancy_distribution(df_subset, df_all):
    """Plot housing occupancy (occupied vs. vacant) for the target blocks.

    Aggregate plot across target blocks plus one plot per individual
    block, with NYC-wide averages overlaid.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks.
    """
    # H3_001N: total housing units. H3_002N occupied, H3_003N vacant.
    total_col = 'H3_001N'
    tenure_cols = ['H3_002N', 'H3_003N']
    labels = ['Occupied', 'Vacant']

    df_subset[[total_col] + tenure_cols] = df_subset[[total_col] + tenure_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_all[[total_col] + tenure_cols] = df_all[[total_col] + tenure_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    total_all = df_all[total_col].sum()
    tenure_total_all = df_all[tenure_cols].sum()
    avg_pct = (tenure_total_all / total_all * 100).values

    total_subset = df_subset[total_col].sum()
    if total_subset > 0:
        tenure_subset_sum = df_subset[tenure_cols].sum()
        subset_pct = (tenure_subset_sum / total_subset * 100).values

        fig, ax = plt.subplots(figsize=(8, 4))
        y = np.arange(len(labels))

        ax.barh(y, subset_pct, color='seagreen', edgecolor='black', label='Hamilton/Howard Beach Census Blocks Total')
        ax.plot(avg_pct, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=11)
        ax.set_xlabel('Percentage of Housing Units', fontsize=12)
        ax.set_title('Housing Occupancy Distribution\nHamilton/Howard Beach Census Blocks', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\allBlocks\\hb_allBlocks_occupancyDistribution.png", dpi=800, bbox_inches='tight')

    # Per-block.
    for _, row in df_subset.iterrows():
        total = row[total_col]
        if total == 0:
            continue

        geo_id = str(row['GEO_ID'])
        block_id = geo_id[-4:]
        values = row[tenure_cols].values.astype(float)
        pct = (values / total * 100)

        fig, ax = plt.subplots(figsize=(8, 4))
        y = np.arange(len(labels))

        ax.barh(y, pct, color='cornflowerblue', edgecolor='black', label=f'Block {block_id}')
        ax.plot(avg_pct, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=11)
        ax.set_xlabel('Percentage of Housing Units', fontsize=12)
        ax.set_title(f'Housing Occupancy Distribution\nBlock {block_id}', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\occupancy\\hb_{block_id}_occupancyDistribution.png", dpi=800, bbox_inches='tight')


plot_occupancy_distribution(df_filtered, df_merged)


def plot_housing_tenure_distribution(df_subset, df_all):
    """Plot housing tenure (owner-occupied vs. renter-occupied).

    H4_002N + H4_003N together cover owner-occupied units (mortgaged
    plus owned-free-and-clear); H4_004N covers renter-occupied units.
    The two owner sub-categories are summed for a binary owner / renter
    split.

    Parameters
    ----------
    df_subset : pandas.DataFrame
        DHC rows for the target census blocks.
    df_all : pandas.DataFrame
        DHC rows for all NYC census blocks.
    """
    total_col = 'H4_001N'
    owner_cols = ['H4_002N', 'H4_003N']
    renter_col = 'H4_004N'
    tenure_cols = owner_cols + [renter_col]
    labels = ['Owner-occupied', 'Renter-occupied']

    df_subset[[total_col] + tenure_cols] = df_subset[[total_col] + tenure_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_all[[total_col] + tenure_cols] = df_all[[total_col] + tenure_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    owner_total_all = df_all[owner_cols].sum().sum()
    renter_total_all = df_all[renter_col].sum()
    total_all = df_all[total_col].sum()
    avg_pct = np.array([owner_total_all, renter_total_all]) / total_all * 100

    # Aggregate across target blocks.
    owner_total_subset = df_subset[owner_cols].sum().sum()
    renter_total_subset = df_subset[renter_col].sum()
    total_subset = df_subset[total_col].sum()

    print(f"--- Combined Housing Tenure for Selected Census Blocks ---")
    print(f"  Owner-occupied: {int(owner_total_subset)}")
    print(f"  Renter-occupied: {int(renter_total_subset)}")
    print(f"Total housing units: {int(total_subset)}\n")

    if total_subset > 0:
        subset_pct = np.array([owner_total_subset, renter_total_subset]) / total_subset * 100

        fig, ax = plt.subplots(figsize=(8, 4))
        y = np.arange(len(labels))

        ax.barh(y, subset_pct, color='seagreen', edgecolor='black', label='Selected Blocks Total')
        ax.plot(avg_pct, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=11)
        ax.set_xlabel('Percentage of Housing Units', fontsize=12)
        ax.set_title('Housing Tenure Distribution\nSelected NYC Census Blocks', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\allBlocks\\hb_allBlocks_housingtenureDistribution.png", dpi=800, bbox_inches='tight')

    # Per-block.
    for _, row in df_subset.iterrows():
        total = row[total_col]
        if total == 0:
            continue

        geo_id = str(row['GEO_ID'])
        block_id = geo_id[-4:]
        owner_count = row[owner_cols].sum()
        renter_count = row[renter_col]
        pct = np.array([owner_count, renter_count]) / total * 100

        print(f"--- Housing Tenure for {block_id} ---")
        print(f"  Owner-occupied: {int(owner_count)}")
        print(f"  Renter-occupied: {int(renter_count)}")
        print(f"Total housing units: {int(total)}\n")

        fig, ax = plt.subplots(figsize=(8, 4))
        y = np.arange(len(labels))

        ax.barh(y, pct, color='cornflowerblue', edgecolor='black', label=f'Block {block_id}')
        ax.plot(avg_pct, y, 'ko', label='NYC Census Block Avg')

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=11)
        ax.set_xlabel('Percentage of Housing Units', fontsize=12)
        ax.set_title(f'Housing Tenure Distribution\nBlock {block_id}', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()
        fig.savefig(f"F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\bar charts\\tenure\\hb_{block_id}_housingtenureDistribution.png", dpi=800, bbox_inches='tight')


plot_housing_tenure_distribution(df_filtered, df_merged)

## Pie Charts of Educational & Household Income Levels:

A side-by-side look at **educational attainment** and **household income** for Hamilton Beach (Queens, green palette) and Red Hook (Brooklyn, blue palette). Both communities sit in FEMA-designated coastal flood zones, but they differ markedly in socioeconomic composition — Hamilton Beach skews toward homeowner-occupied middle-income households, Red Hook toward a more income-bimodal renter population — and the pie charts make that contrast immediately visible.

The underlying data is IPUMS NHGIS `nhgis0004_ds267_20235` (ACS 5-year, block-group resolution). Education categories are collapsed from the 24 ACS bins into 9 readable buckets (No HS / Only HS / GED / Incomplete College / Associate / Bachelor / Master / Professional / Doctorate); income from 16 ACS bins into 9 buckets (`<$10k` through `$200k+`).

In [ ]:
"""
py_chart.py
===========
Side-by-side educational-attainment and household-income pie charts for
the Hamilton Beach and Red Hook focus communities.

Uses the IPUMS NHGIS ``nhgis0004_ds267_20235`` ACS 5-year extract at
block-group resolution. Educational attainment is collapsed from the 24
ACS bins (``ASP3E002``..``ASP3E025``) into 9 readable categories.
Household income is collapsed from 16 ACS bins
(``ASQOE002``..``ASQOE017``) into 9 income brackets.

Outputs (written to ``plots/demographics/pi charts/``):

- ``hamBeach_pi.png``: Hamilton Beach education + income, green palette.
- ``redHook_pi.png``: Red Hook education + income, blue palette.

The two output figures use a consistent legend / category ordering so
the distributions are directly comparable side by side.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch

# Load the ACS block-group extract.
df = pd.read_csv(
    r'F:\dsl_CLIMA\projects\submittable\clima\source\census demographic files\nhgis0004_ds267_20235_blck_grp.csv',
    dtype={'TL_GEO_ID': str}
)

# Target block-group GEOIDs.
# Hamilton Beach is a single block group (Queens, NY).
# Red Hook spans 11 block groups in Brooklyn, NY.
ham_beach = ['360810884006']
red_hook = [
    '360470053031', '360470053011', '360470053012', '360470053022',
    '360470085003', '360470059002', '360470059001', '360470085001',
    '360470085002', '360470053021', '360470053023'
]

ham_df = df[df['TL_GEO_ID'].isin(ham_beach)]
red_df = df[df['TL_GEO_ID'].isin(red_hook)]

# ----------------------------------------------------------------------
# Raw ACS bin column lists.
# - ASP3E002..ASP3E025: 24 educational-attainment bins for age 25+.
# - ASQOE002..ASQOE017: 16 household-income brackets.
# ----------------------------------------------------------------------
edu_cols = [f"ASP3E{str(i).zfill(3)}" for i in range(2, 26)]
income_cols = [f"ASQOE{str(i).zfill(3)}" for i in range(2, 18)]

ham_edu_dist = ham_df[edu_cols].sum()
red_edu_dist = red_df[edu_cols].sum()
ham_income_dist = ham_df[income_cols].sum()
red_income_dist = red_df[income_cols].sum()

# ----------------------------------------------------------------------
# Education bucket mapping: collapse the 24 ACS bins into 9 readable
# categories. Indices reference positions within ``edu_cols``.
# ----------------------------------------------------------------------
simplified_edu_groups = {
    "No HS Diploma": list(range(0, 15)),
    "Only HS Diploma": [15],
    "GED": [16],
    "Incomplete College Degree": [17, 18],
    "Associate": [19],
    "Bachelor": [20],
    "Master": [21],
    "Professional": [22],
    "Doctorate": [23]
}


def simplify_education_distribution(data):
    """Collapse the 24 ACS education bins into 9 readable categories.

    Parameters
    ----------
    data : pandas.Series
        Series with one entry per raw ASP3E bin (length 24).

    Returns
    -------
    pandas.Series
        Series with one entry per simplified bucket, indexed by the
        bucket label.
    """
    simplified = {}
    for label, indices in simplified_edu_groups.items():
        simplified[label] = data.iloc[indices].sum()
    return pd.Series(simplified)


ham_edu_simple = simplify_education_distribution(ham_edu_dist)
red_edu_simple = simplify_education_distribution(red_edu_dist)

# ----------------------------------------------------------------------
# Income bucket mapping: collapse the 16 ACS bins into 9 brackets.
# Indices reference positions within ``income_cols``.
# ----------------------------------------------------------------------
merged_income_labels = [
    "<$10k", "$10k-25k", "$25k-35k", "$35k-45k", "$45k-60k",
    "$60k-85k", "$100k-150k", "$150k-200k", "$200k+"
]

merged_income_groups = {
    "<$10k": [0],                    # ASQOE002
    "$10k-25k": [1, 2, 3],           # ASQOE003-ASQOE005
    "$25k-35k": [4, 5],              # ASQOE006-ASQOE007
    "$35k-45k": [6, 7],              # ASQOE008-ASQOE009
    "$45k-60k": [8, 9],              # ASQOE010-ASQOE011
    "$60k-85k": [10, 11],            # ASQOE012-ASQOE013
    "$100k-150k": [12, 13],          # ASQOE014-ASQOE015
    "$150k-200k": [14],              # ASQOE016
    "$200k+": [15]                   # ASQOE017
}


def simplify_income_distribution(data):
    """Collapse the 16 ACS income bins into 9 brackets.

    Parameters
    ----------
    data : pandas.Series
        Series with one entry per raw ASQOE bin (length 16).

    Returns
    -------
    pandas.Series
        Series with one entry per income bracket, indexed by bracket
        label.
    """
    simplified = {}
    for label, indices in merged_income_groups.items():
        simplified[label] = data.iloc[indices].sum()
    return pd.Series(simplified)


ham_income_simple = simplify_income_distribution(ham_income_dist)
red_income_simple = simplify_income_distribution(red_income_dist)


def plot_pie(data, labels, title, palette='Blues', ax=None, radius=1.6):
    """Render a single annotated pie chart with percentage labels.

    Slices are ordered along a fixed logical ordering (smallest-to-
    largest education tier or smallest-to-largest income bracket) so
    color and label position are consistent across plots that share a
    palette. Zero-count categories are omitted from the pie but kept in
    the legend so the visual key matches across communities.

    Parameters
    ----------
    data : pandas.Series
        Counts indexed by the bucket labels in ``labels``.
    labels : list of str
        Full list of bucket labels expected in ``data``.
    title : str
        Subplot title.
    palette : str, default 'Blues'
        Matplotlib colormap name. Hamilton Beach uses 'Greens', Red
        Hook uses 'Blues'.
    ax : matplotlib.axes.Axes, optional
        Axis to draw on. If None, a new figure is created.
    radius : float, default 1.6
        Pie radius in data units.
    """
    data = data.fillna(0)

    if len(data) != len(labels):
        print(f"Warning: Mismatch between data and labels for {title}")
        return

    # Logical orderings for the two figure types. Hispanic and "all"
    # buckets are filtered out automatically because they don't appear
    # in either list.
    edu_order = [
        "No HS Diploma", "Only HS Diploma", "GED", "Incomplete College Degree",
        "Associate", "Bachelor", "Master", "Professional", "Doctorate"
    ]
    income_order = [
        "<$10k", "$10k-25k", "$25k-35k", "$35k-45k", "$45k-60k",
        "$60k-85k", "$100k-150k", "$150k-200k", "$200k+"
    ]
    if all(label in edu_order for label in labels):
        logical_order = edu_order
    elif all(label in income_order for label in labels):
        logical_order = income_order
    else:
        logical_order = sorted(labels)

    # Build a stable label-to-color mapping so the same bucket always
    # gets the same color across panels.
    cmap = cm.get_cmap(palette, len(logical_order))
    label_color_map = {label: cmap(i) for i, label in enumerate(logical_order)}

    # Slice the pie only over non-zero buckets so we don't emit
    # invisible wedges that still consume legend space.
    data_nonzero = data[data > 0]
    sorted_labels = [label for label in logical_order if label in data_nonzero.index]
    sorted_data = [data[label] for label in sorted_labels]
    sorted_colors = [label_color_map[label] for label in sorted_labels]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    else:
        fig = ax.figure

    if not sorted_data:
        ax.text(0.5, 0.5, "No Data", ha="center", va="center", fontsize=14)
        ax.axis('off')
        return

    wedges, _ = ax.pie(
        sorted_data,
        colors=sorted_colors,
        startangle=0,
        labels=None,
        radius=radius,
        wedgeprops=dict(edgecolor='white')
    )

    total = sum(sorted_data)

    # Annotate each wedge with its percentage, anchored by a thin
    # leader line so labels don't collide with the pie edge.
    for wedge, val in zip(wedges, sorted_data):
        angle = (wedge.theta2 + wedge.theta1) / 2
        x = np.cos(np.radians(angle))
        y = np.sin(np.radians(angle))
        ha = 'left' if x > 0 else 'right'
        pct = f"{100 * val / total:.1f}%"

        label_pos = radius * 1.15
        arrow_start = radius * 0.75

        ax.annotate(
            pct,
            xy=(x * arrow_start, y * arrow_start),
            xytext=(x * label_pos, y * label_pos),
            ha=ha,
            va='center',
            fontsize=18,
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
            color='black'
        )

    # Build the legend from the full bucket list (not just non-zero) so
    # zero-count categories still appear in the visual key.
    legend_labels = [label for label in logical_order if label in labels]
    legend_patches = [
        Patch(facecolor=label_color_map[label], edgecolor='none') for label in legend_labels
    ]

    ax.legend(
        legend_patches,
        legend_labels,
        loc='upper center',
        bbox_to_anchor=(0.5, 0),
        ncol=3,
        fontsize=13,
        labelcolor='black'
    )

    ax.set_title(title, fontsize=24, pad=30)
    ax.axis('equal')


edu_labels = [
    "No HS", "Only HS", "GED", "Incomplete College Degree",
    "Associate", "Bachelor", "Master", "Professional", "Doctorate"
]
income_labels = [
    "<$10k", "$10k-25k", "$25k-35k", "$35k-45k", "$45k-60k",
    "$60k-85k", "$100k-150k", "$150k-200k", "$200k+"
]

# ----------------------------------------------------------------------
# Hamilton Beach: education + income side by side, green palette.
# ----------------------------------------------------------------------
fig_ham, axes_ham = plt.subplots(1, 2, figsize=(14, 7))
plot_pie(ham_edu_simple, ham_edu_simple.index.tolist(), "Educational Level", palette='Greens', ax=axes_ham[0])
plot_pie(ham_income_simple, ham_income_simple.index.tolist(), "Household Income", palette='Greens', ax=axes_ham[1])
fig_ham.suptitle('Hamilton Beach Demographics', fontsize=40, x=.525)

plt.tight_layout()
plt.show()
fig_ham.savefig('F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\pi charts\\hamBeach_pi.png', dpi=800, bbox_inches='tight')

# ----------------------------------------------------------------------
# Red Hook: education + income side by side, blue palette. The category
# axes are kept identical to the Hamilton Beach figure so the two are
# directly comparable.
# ----------------------------------------------------------------------
fig_red, axes_red = plt.subplots(1, 2, figsize=(14, 7))
plot_pie(red_edu_simple, red_edu_simple.index.tolist(), "Educational Level", palette='Blues', ax=axes_red[0])
plot_pie(red_income_simple, red_income_simple.index.tolist(), "Household Income", palette='Blues', ax=axes_red[1])
fig_red.suptitle('Red Hook Demographics', fontsize=40, x=.525)

plt.tight_layout()
plt.show()
fig_red.savefig('F:\\dsl_CLIMA\\projects\\submittable\\clima\\plots\\demographics\\pi charts\\redHook_pi.png', dpi=800, bbox_inches='tight')

---
# Results & Discussion
---

## Scaling Estimates Table

The fitted $\beta$ values across all four GEOID levels and all three network types are summarized below. CIs are 95% via the $t$-distribution with $N-2$ degrees of freedom.

| GEOID Type | GEOIDs | Network Type | β (Scaling Exponent) | γ (Intercept) | Exp. CI       | Intercept CI  | Adj. R² | Rescaled Degree | Avg Pop Est. | Avg 2022 User Est. |
|------------|--------|--------------|-------------------|---------------|---------------|---------------|---------|----------------|--------------|------------------|
| County     | 3141   | Total        | 0.972             | 0.019         | [0.967,0.977] | [0.015,0.024] | 0.979   | 115,046,176    |              |                  |
| County     | 3141   | Inner        | 1.062             | -0.013        | [1.052,1.073] |[-0.022,-0.004]| 0.925   | 35,816,319     | 106,879      | 54,429           |
| County     | 3141   | Outgoing     | 0.945             | 0.024         | [0.939,0.950] | [0.020,0.029] | 0.975   | 79,229,857     |              |                  |
|------------|--------|--------------|-------------------|---------------|---------------|---------------|---------|----------------|--------------|------------------|
| CBSA       | 917    | Total        | 0.979             | 0.009         | [0.967,0.990] | [0.000,0.017] | 0.969   | 369,646,071    |              |                  |
| CBSA       | 917    | Inner        | 1.064             | -0.065        | [1.046,1.082] |[-0.079,-0.051]| 0.937   | 166,674,627    | 345,975      | 175,799          |
| CBSA       | 917    | Outgoing     | 0.924             | 0.047         | [0.913,0.936] | [0.039,0.056] | 0.966   | 202,971,443    |              |                  |
|------------|--------|--------------|-------------------|---------------|---------------|---------------|---------|----------------|--------------|------------------|
| MSA        | 381    | Total        | 0.982             | 0.006         | [0.962,1.002] | [-0.006,0.017]| 0.962   | 806,430,257    |              |                  |
| MSA        | 381    | Inner        | 1.082             | -0.048        | [1.049,1.114] |[-0.067,-0.028]| 0.919   | 372,823,802    | 760,791      | 385,411          |
| MSA        | 381    | Outgoing     | 0.914             | 0.033         | [0.894,0.933] | [0.021,0.045] | 0.958   | 433,696,455    |              |                  |
|------------|--------|--------------|-------------------|---------------|---------------|---------------|---------|----------------|--------------|------------------|
| muSA       | 536    | Total        | 0.969             | -0.009        | [0.931,1.007] | [-0.018,0.000]| 0.821   | 59,170,744     |              |                  |
| muSA       | 536    | Inner        | 1.054             | -0.029        | [0.994,1.113] |[-0.042,-0.016]| 0.694   | 20,139,486     | 51,115       | 26,802           |
| muSA       | 536    | Outgoing     | 0.921             | -0.007        | [0.883,0.960] | [-0.016,0.002]| 0.805   | 39,031,258     |              |                  |
|------------|--------|--------------|-------------------|---------------|---------------|---------------|---------|----------------|--------------|------------------|

### Reading the scaling table

* **Inner connections are superlinear** at every GEOID level — $\beta_{\text{Inner}} \in [1.054, 1.082]$ — and the entire 95% CI sits *above* 1 at the County, CBSA, and MSA levels. This is the same qualitative finding as *Schläpfer et al.* (2014) for face-to-face / phone interactions within cities: the larger the place, the *more than proportionally* its residents interact with each other. Our point estimate at the County level ($\beta = 1.062$) is at the low end of what they report for European urban areas (typically $\beta \approx 1.1\!-\!1.2$), which we read as a real but modest superlinearity at U.S. county resolution.

* **Outgoing connections are sublinear** at every level — $\beta_{\text{Outgoing}} \in [0.914, 0.945]$ — and again the 95% CI excludes 1 at the County, CBSA, and MSA levels. Larger places generate *proportionally fewer* outgoing per-capita ties, which is consistent with denser local social networks crowding out long-range ties as a place grows.

* **Total connections sit just below 1** — $\beta_{\text{Total}} \in [0.969, 0.982]$ — i.e., the superlinearity of the inner ties is approximately, but not quite, cancelled by the sublinearity of the outgoing ties. The Total exponent is the closest of the three to exact linearity, which is intuitive: once you sum within and across, the total degree of a place should be *roughly* proportional to its population.

* **muSA fits are noisier** ($R^2 \approx 0.69\!-\!0.82$) than MSA fits ($R^2 \approx 0.92\!-\!0.96$). This is expected — micropolitan areas are smaller and span a narrower range of population, so the regression has less leverage. The muSA Total CI ($[0.931, 1.007]$) actually *includes* 1.0, so we cannot reject pure linearity for total connectivity in the micropolitan subsample.

* **Stability across GEOID types**: the fact that $\beta_{\text{Inner}}$, $\beta_{\text{Outgoing}}$, and $\beta_{\text{Total}}$ stay within a tight band as we change GEOID resolution (County $\to$ CBSA $\to$ MSA) is a sanity check on the rescaling procedure — the exponents are not artifacts of how we drew the geographic partition.

---

### What the histograms told us

Before fitting any regression, the connection / user / population histograms in the **Distributions** section confirm the assumptions we need:

* The **log-transformed** Inter-, Outgoing-, and Total-connection counts at all four GEOID levels are close to Normal in shape — i.e., the raw counts are roughly **log-normal**, which is the canonical distribution arising from multiplicative growth processes. This is precisely what justifies treating $\epsilon_i$ in the Basic Power Law as multiplicative noise and fitting in log-log space.
* The fitted Log-Normal PDF has the lowest RMSE vs. KDE for the bulk of variables, with the SkewNormal occasionally edging it out for the more right-skewed populations.
* The Generalized Pareto fit to the top 75% tail captures the heavy upper tail (the few outsize metros that dominate the absolute connection counts) without forcing the bulk distribution into a heavy-tailed family.

### What the choropleths told us

* The **raw connection choropleths** light up exactly where expected — BosWash, Chicago, the Bay Area + LA, Texas Triangle, Front Range, Florida — confirming that the absolute connection counts track population.
* The **coverage choropleth** ($s_i = |S_i|/N_i$) is more revealing: coverage spans roughly $0.34\!-\!0.63$ across counties, with higher coverage in dense, younger, more-online counties (urban Northeast, Pacific coast, parts of the Mountain West) and lower coverage in much of the rural South and Plains. This is exactly the kind of *geographically heterogeneous undercount* the *Schläpfer* rescaling step is designed to absorb.
* The **per-user choropleths** — connections divided by Facebook MAU — are visibly **flatter** than the raw-count choropleths. That's the qualitative confirmation that rescaling by $s_i$ has done its job: once we strip out the population-driven baseline, per-user connectivity is no longer dominated by where the people are.

### What the demographic figures told us

The Census-block analyses in the Demographics section sit at a completely different scale from the national regressions, but they're the bridge to the CLIMA application:

* **Hamilton + Howard Beach** (Queens) skews older than the NYC average, is ~63% owner-occupied vs. ~51% citywide, and has a household-size distribution shifted toward 3-4-person households. Education is bottom-heavy: ~30% no HS diploma, ~26% only HS, ~24% bachelor's-or-higher.
* **Red Hook** (Brooklyn) is markedly different — more bachelor's-and-above (~32%), a notably bimodal income distribution with a much fatter upper tail (~30% over $100k), and a different racial composition. The two communities sit in the same FEMA flood-zone designations but represent very different *social* exposures.
* These contrasts are the reason CLIMA's modeling can't treat "coastal homeowner" as a uniform population class — the same flood event will produce very different decision cascades depending on tenure, income, and community structure.



---

## Future Work
* **Mobility model with flood-risk benefits.** After reading [*A Universal Model for Mobility and Migration Patterns* (Simini et al., 2012)](https://doi.org/10.1038/nature10856), we began formulating a model using FEMA's National Risk Index as part of the benefit-distribution term, with the *ratio of available homes* (rather than the radiation model's ratio of available jobs) as the relevant opportunity surface for homeowner relocation decisions. The scaling exponents in this notebook are an empirical input to that model — they parameterize how *fast* information about flood events can propagate through a community of a given size.
* **Re-attempt community identification.** The Clique Percolation Method (CPM) on the symmetric ~10M-row edge list exhausted available RAM mid-build. A memory-frugal variant — e.g., Louvain on the bipartite county-user projection, or CPM restricted to within-CBSA edges — should be tractable on a higher-memory machine and would give us a third lens (mesoscale community structure) on top of the scaling and demographic analyses.
* **Broaden the demographic comparison.** Extend the block-level demographic snapshots beyond Hamilton Beach / Howard Beach / Red Hook to the full set of NYC waterfront census tracts under FEMA 100-year and 500-year designations, so the scaling and demographic strands can be joined into a single block-level vulnerability picture.
* **Cross-validate ESRI MAU.** ESRI's `MP19049a_B` is a model output; the most natural cross-check would be against state-level Pew Research surveys on Facebook adoption rates, which would let us calibrate the ~17% gap between our average $s_i$ and Meta's published MAU coverage figure.

## Potential Problems
* **County-level Facebook MAU is not publicly reported by Meta.** ESRI is our best stand-in, but comparing our derived average $s_i$ against Meta's published Q4 '22 MAU figure suggests the ESRI-derived coverage runs ~17% below the true MAU coverage. A uniform underestimate would shift the intercept $\gamma$ but leave $\beta$ unchanged in expectation; however, because ESRI's MAU model is itself geographically heterogeneous (undercounting more in certain county types than others), there is a second-order effect on $\beta$ that's harder to bound. The most plausible direction is a small *decrease* in $\beta$ — i.e., the true scaling is slightly *more* sublinear than what we report for Outgoing and Total.
* **SCI itself is noised.** Gaussian noise in $\pm[0,1]$ is added by Meta to the raw connection counts to preserve privacy. For large counties this is invisible, but for the lowest-user counties (just above the 50,000-user reporting threshold) it can dominate the signal — this is part of why the muSA confidence intervals are wider.
* **ESRI MAU is a model output**, not a direct count — so two layers of estimation (Meta's true MAU → ESRI's modeled MAU → our $|S_i|$) sit between the underlying truth and the regressor.
* **The demographic figures are descriptive.** Three communities ≠ a generalizable claim about coastal NYC. We use these blocks to ground the modeling work in concrete cases, not to extrapolate to broader populations.
---

## References

### Meta SCI Resources
- [SCI Homepage](https://data.humdata.org/dataset/social-connectedness-index)
- [SCI Methodology](https://dataforgood.facebook.com/dfg/docs/methodology-social-connectedness-index)
- [SCI Docs](https://data.humdata.org/dataset/e9988552-74e4-4ff4-943f-c782ac8bca87/resource/a0c37eb4-b45c-436d-b2b2-c0c9b1974318/download/documentation-fb-social-connectedness-index-october-2021.pdf)
- [County-to-County SCI Dataset](https://data.humdata.org/dataset/e9988552-74e4-4ff4-943f-c782ac8bca87/resource/c59fd5ac-0458-4e83-b6be-5334f0ea9a69/download/us-counties-us-counties-fb-social-connectedness-index-october-2021.zip)

### Meta Official Regional Coverage Estimates
- [Meta Q4 '22 Earnings Presentation](https://s21.q4cdn.com/399680738/files/doc_financials/2023/q4/Earnings-Presentation-Q4-2023.pdf)

### ESRI Facebook User Estimates
- [ESRI Data](https://nyuds.maps.arcgis.com/home/item.html?id=14a2fb32e22b4fe5ab9d884c9e994075)
- [ESRI Documentation](https://demographics5.arcgis.com/arcgis/rest/services/USA_MPI_1_2022/MapServer/7)

### Crosswalk
- [County-MSA-CSA Crosswalk](https://www.bls.gov/cew/classifications/areas/county-msa-csa-crosswalk.html)

### Census / Demographic Data
- [IPUMS NHGIS — 2020 DHC and ACS extracts](https://www.nhgis.org/)
- [TIGER/Line Shapefiles (2021)](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html)

## Bibliography

1. Bailey, Michael, Rachel Cao, Theresa Kuchler, Johannes Stroebel, and Arlene Wong.
   **"Social Connectedness: Measurement, Determinants, and Effects."**
   *Journal of Economic Perspectives* 32, no. 3 (August 2018): 259-280.
   DOI: [10.1257/jep.32.3.259](https://doi.org/10.1257/jep.32.3.259)

2. Schläpfer, M., Bettencourt, L. M. A., Grauwin, S., Raschke, M., Claxton, R., Smoreda, Z., West, G. B., & Ratti, C. (2014).
   **The scaling of human interactions with city size.**
   *Journal of the Royal Society Interface*, **11**(98), 20130789.
   DOI: [10.1098/rsif.2013.0789](https://doi.org/10.1098/rsif.2013.0789)

3. Simini, F., González, M., Maritan, A., et al.
   **A universal model for mobility and migration patterns.**
   *Nature* **484**, 96-100 (2012).
   DOI: [10.1038/nature10856](https://doi.org/10.1038/nature10856)
---
al fin :]
